# dWGS — Part 8: Data Requirements (NEY GMS → NW GMS, RGL to SGL)

NHS England is transitioning centralised Whole Genome Sequencing (WGS) to a distributed
model (dWGS): each NHS GMS geography's **Requesting Genomic Laboratory (RGL)** submits
DNA samples directly to a **Sequencing Genomic Laboratory (SGL)**. This notebook covers
a specific subcontracted arrangement: **North East and Yorkshire GMS acting as RGL**,
submitting samples and a digital manifest to **NW GMS acting as SGL**, whose inbound
order pathway is the same NW Regional Integration Engine (RIE) → iGene ORM^O01
interface `03`-`07` already work with.

Three source documents define the requirement (all in `NotGit/`, gitignored per this
repo's convention, referenced here by name only):

- **`RGL to SGL SOP v1.0 (DRAFT).docx`** — NHS England's SOP; Appendix 3 defines the
  42-column national Digital Manifest CSV. Items 38-42 (patient name, family structure,
  participant type and clinical information) are specific to the NE&Y➜NW pathway.
- **`dWGS Manifest CSV to HL7v2 ORM Transformation and iGene Import Specification
  v0.5`** — NW GMS's draft spec turning that manifest into an ORM^O01 message for iGene
  (circulated as `dWGS Sample Manifest HL7v2 Data Mapping v0.4.pdf`).
- **`dWGS Sample Manifest HL7v2 Data Mapping v0.4.xlsx`** — the field-by-field working
  sheet those tables were drawn from.

This is a preparation notebook: the sample data (`Input/dWGS.csv`) and the three-way
field mapping below.

## Where this sits: from clinical order to sequencing result

This notebook's `LAB-35` sub-order is one hop in a longer chain that starts and ends
with the referring clinician. Mapped onto [NW-GMSA's Inter-Laboratory Workflow
(ILW)](https://nw-gmsa.github.io/en/ILW.html#sub-orders-lab-35-and-lab-36) actors:

| Role (ILW) | Party here | Transaction it sends |
|---|---|---|
| Order Placer | Test Ordering Entity (referring Trust/clinician) | `LAB-1` |
| Order Filler | Requesting Genomic Laboratory (RGL) | `LAB-35` (as sub-order placer), `LAB-3` |
| Sub Contractor | Sequencing Genomic Laboratory (SGL) | `LAB-36` *(Option B, below)* |
| Automation Manager | Genomics England Limited (GEL) | `LAB-5` *(Option A, below)* |

`LAB-1` (the original clinical order) and `LAB-35` (RGL's sub-order to the SGL) are
settled — that's what this notebook builds. What's still open is how the result of
that sub-order gets back to the RGL, because GEL sits behind the SGL in the physical
pathway (RGL to SGL SOP v1.0, Figure 1) and doesn't map cleanly onto a single ILW role.

### Process flow

![notebook 8 diagram 1](https://mermaid.ink/svg/Zmxvd2NoYXJ0IFRECiAgICBPUFsiT3JkZXIgUGxhY2VyXG5UZXN0IE9yZGVyaW5nIEVudGl0eSJdCiAgICBPRlsiT3JkZXIgRmlsbGVyXG5SZXF1ZXN0aW5nIEdlbm9taWMgTGFib3JhdG9yeSAoUkdMKSJdCgogICAgc3ViZ3JhcGggU0VRWyJTZXF1ZW5jaW5nIl0KICAgICAgICBTQ1siU3ViIENvbnRyYWN0b3JcblNlcXVlbmNpbmcgR2Vub21pYyBMYWJvcmF0b3J5IChTR0wpIl0KICAgICAgICBHRUxbIkdlbm9taWNzIEVuZ2xhbmQgTGltaXRlZFxuKEF1dG9tYXRpb24gTWFuYWdlcikiXQogICAgICAgIFNDIC0tICJwcmVwYXJlZCBzYW1wbGUiIC0tPiBHRUwKICAgIGVuZAoKICAgIE9QIC0tICJMQUItMVxubGFib3JhdG9yeSBvcmRlciIgLS0+IE9GCiAgICBPRiAtLSAiTEFCLTM1XG5zdWItb3JkZXIgKyBtYW5pZmVzdCIgLS0+IFNDCiAgICBHRUwgLS4gIkxBQi01IChPcHRpb24gQSlcbmRpcmVjdCB0byBPcmRlciBGaWxsZXIiIC4tPiBPRgogICAgU0MgLS4gIkxBQi0zNiAoT3B0aW9uIEIpXG52aWEgU3ViIENvbnRyYWN0b3IiIC4tPiBPRgogICAgT0YgLS0gIkxBQi0zXG5sYWJvcmF0b3J5IHJlcG9ydCIgLS0+IE9Q)

<!--
```mermaid
flowchart TD
    OP["Order Placer\nTest Ordering Entity"]
    OF["Order Filler\nRequesting Genomic Laboratory (RGL)"]

    subgraph SEQ["Sequencing"]
        SC["Sub Contractor\nSequencing Genomic Laboratory (SGL)"]
        GEL["Genomics England Limited\n(Automation Manager)"]
        SC -- "prepared sample" --> GEL
    end

    OP -- "LAB-1\nlaboratory order" --> OF
    OF -- "LAB-35\nsub-order + manifest" --> SC
    GEL -. "LAB-5 (Option A)\ndirect to Order Filler" .-> OF
    SC -. "LAB-36 (Option B)\nvia Sub Contractor" .-> OF
    OF -- "LAB-3\nlaboratory report" --> OP
```
-->

### Sequence diagram

![notebook 8 diagram 2](https://mermaid.ink/svg/c2VxdWVuY2VEaWFncmFtCiAgICBhY3RvciBPUCBhcyBPcmRlciBQbGFjZXI8YnIvPihUZXN0IE9yZGVyaW5nIEVudGl0eSkKICAgIHBhcnRpY2lwYW50IE9GIGFzIE9yZGVyIEZpbGxlcjxici8+KFJHTCkKICAgIHBhcnRpY2lwYW50IFNDIGFzIFN1YiBDb250cmFjdG9yPGJyLz4oU0dMKQogICAgcGFydGljaXBhbnQgR0VMIGFzIEdlbm9taWNzIEVuZ2xhbmQgTHRkPGJyLz4oQXV0b21hdGlvbiBNYW5hZ2VyKQoKICAgIE9QLT4+T0Y6IExBQi0xIExhYm9yYXRvcnkgT3JkZXIKICAgIE9GLT4+U0M6IExBQi0zNSBTdWItb3JkZXIgKHNhbXBsZSArIGRpZ2l0YWwgbWFuaWZlc3QpCiAgICBTQy0+PkdFTDogUmF3IGdlbm9taWMgZGF0YSAocG9zdC1zZXF1ZW5jaW5nKQoKICAgIGFsdCBPcHRpb24gQSDigJQgR0VMIGFzIEF1dG9tYXRpb24gTWFuYWdlcgogICAgICAgIEdFTC0+Pk9GOiBMQUItNSBzdHJ1Y3R1cmVkIHJlc3VsdCAoZGlyZWN0IHRvIE9yZGVyIEZpbGxlcikKICAgIGVsc2UgT3B0aW9uIEIg4oCUIHJlc3VsdCByZXR1cm5lZCB2aWEgdGhlIFN1YiBDb250cmFjdG9yCiAgICAgICAgR0VMLT4+U0M6IEFuYWx5c2VkL2ludGVncmF0ZWQgcmVzdWx0CiAgICAgICAgU0MtPj5PRjogTEFCLTM2IFN1Yi1vcmRlciByZXN1bHQKICAgIGVuZAoKICAgIE9GLT4+T1A6IExBQi0zIExhYm9yYXRvcnkgcmVwb3J0)

<!--
```mermaid
sequenceDiagram
    actor OP as Order Placer<br/>(Test Ordering Entity)
    participant OF as Order Filler<br/>(RGL)
    participant SC as Sub Contractor<br/>(SGL)
    participant GEL as Genomics England Ltd<br/>(Automation Manager)

    OP->>OF: LAB-1 Laboratory Order
    OF->>SC: LAB-35 Sub-order (sample + digital manifest)
    SC->>GEL: Raw genomic data (post-sequencing)

    alt Option A — GEL as Automation Manager
        GEL->>OF: LAB-5 structured result (direct to Order Filler)
    else Option B — result returned via the Sub Contractor
        GEL->>SC: Analysed/integrated result
        SC->>OF: LAB-36 Sub-order result
    end

    OF->>OP: LAB-3 Laboratory report
```
-->

The rest of this notebook zooms into the `LAB-35` step: NE&Y Genomics acting as RGL
(the *Order Filler* above, confusingly also acting as *sub-order placer* toward NW
Genomics), and NW Genomics acting as SGL (*Sub Contractor* above). The field mapping,
worked example, and message-exchange sections below are all about that one hop, not
the full pathway shown here.

## Field mapping: CSV → HL7v2 → FHIR

All 42 manifest fields from `RGL to SGL SOP v1.0` Appendix 3, in column order. The first
37 are the national core; items 38-42 (`patient_forename`, `patient_surname`,
`family_structure`, `participant_type`, `clinical_information`) were requested by NW GMS
and added by NHS England for the NE&Y➜NW pathway (the `source` column marks them).
`Input/dWGS.csv` carries all 42. HL7v2 mappings follow the transformation spec v0.5.
Fields with no mapping are left blank in both the `hl7v2_field` and `fhir_field` columns —
they're carried in the CSV but not transmitted onward.

NHS England won't add the other two fields NW GMS asked for, `placer_order_number` and
`patient_medical_record_number`. The spec v0.5 uses `dispatched_sample_lsid` as the
interim Placer Order Number and `patient_ngis_id` as the MRN-equivalent identifier
instead. This notebook follows it for the LSID, but sends `patient_ngis_id` as `PI`, not
`MR` - see [Patient](#Patient) below.

The `fhir_field` column is this notebook's own addition — neither source document
mentions FHIR. Identifiers/profiles reused from `03`-`07` are shown as-is.
`referral_id` identifies the NGIS referral, modelled as a **parent `ServiceRequest`**
(`intent` `original-order`) with `referral_id` as its Order Filler Number (`FILL`, system
`https://genomicsengland.co.uk/healthcare/referral-id`). The sub-order points
at it with `basedOn`, and carries the same value as its `requisition`. The parent's
`subject` is the referral's family: a logical reference to a `Group` identified by
`referral_id`. `family_structure` and `participant_type` are ask-at-order-entry answers,
each an [`ObservationOrder`](https://nw-gmsa.github.io/en/StructureDefinition-ObservationOrder.html)
on the order it describes: `family_structure` describes the referral, so it's referenced
from the parent's `supportingInfo`; `participant_type` describes this participant, so it's
referenced from the sub-order's `supportingInfo`. Both are about the `Patient` (the IG's
`ObservationOrder` profile only allows a `Patient` subject).

**Two specimens, not one**: the `primary_sample_*` fields describe the specimen as
originally received at the GLH (blood/tissue, before extraction); `dispatched_sample_*`
describes the extracted DNA sent onward. Each is its own `SPM` segment/`Specimen`
resource - the dispatched specimen's `Specimen.parent` would reference the primary
specimen's. `primary_sample_type`'s germline/tumour distinction has no clean SPM/Specimen
field to sit in (HL7v2 has no field for it, and FHIR's closer fit is arguably an
`Observation` component, not `Specimen`) - marked low-confidence rather than guessed.

**GLH codes are not trust codes, or ODS codes**: `ordering_entity_id` is correctly a
referring NHS Trust's ODS code (e.g. `RR8` Leeds Teaching Hospitals) — but
`glh_laboratory_id` is Genomics England's own **short GLH code**, not an ODS code at all.
A real NW GLH manifest export (`ngis_glh_to_gel_sample_sent_now_<timestamp>.csv`, kept in
`NotGit/`) carries `now` for North West in every row, and the data mapping workbook gives
`now` as its example too. `Input/dWGS.csv` uses `yne` for North East and Yorkshire (not
yet confirmed against a real NE&Y export). Wherever FHIR or v2 needs an ODS code (message
sender, requester, identifier assigners), the worked example looks it up from the GLH
code (`GEL_GLH_ODS` below). NE&Y's hub ODS code, confirmed against the live NHS ODS API
(`directory.spineservices.nhs.uk`), is **`699N0`, "YORKSHIRE AND NORTH EAST GLH"**
(Newcastle-based). It's an ODS tier above individual trusts, the same pattern
`02-work-orders-worked-example.ipynb` explains for `699X0`/`K1S6S`, North West's own
hub/site codes. The `Questionnaire-dWGSSubOrder` design note still calls
`glh_laboratory_id` an ODS code, which the real export contradicts.

**Sample type and state are GEL enumerations, not SNOMED CT**: the same real export and
workbook use `primary_sample_type`/`dispatched_sample_type` values such as
`Normal or Germline sample`, `Solid tumour sample` and `Liquid tumour sample`, and
`primary_sample_state` values such as `blood_unsorted_edta` (`dispatched_sample_state` is
`DNA`). In the GEL Report Models participant schema 1.3.0
([`CommonParticipant.avdl`](https://github.com/genomicsengland/GelReportModels/blob/master/schemas/IDLs/org.gel.models.participant.avro/1.3.0/CommonParticipant.avdl)),
`dispatched_sample_state` is the `Product` enumeration (`DNA`, `RNA`), and
`primary_sample_state` is the TOMS `sampleState`: free text "as entered by the clinician
in TOMs", not an enumeration. Its values combine three GEL enumerations, so the worked
example splits each one (`GEL_SAMPLE_STATE` below), e.g. `blood_unsorted_edta`:

| Part | GEL enumeration | FHIR |
|---|---|---|
| `blood` | `SampleSource` ("The source of the sample") | `Specimen.type` - GEL coding plus a SNOMED CT code from NW-GMSA's specimen-type ValueSet (`GEL_SAMPLE_SOURCE_SNOMED`), which the IG's required binding needs |
| `unsorted` | `Method` ("how the sample was taken directly from the patient") | `Specimen.collection.method` |
| `edta` | `StorageMedium` ("what solvent/medium the sample was stored in") | `Specimen.processing` |

The original value is kept as `Specimen.type.text`. The split is inferred from the values
seen so far (`tumour_freshFrozen` has no `Method` part); GEL doesn't publish the full
`sampleState` list, or FHIR CodeSystems for its enumerations, so the GEL system URIs are
provisional. `received_sample_topography`, not `primary_sample_state`, is the body site
(`Specimen.collection.bodySite`). `StorageMedium` goes in `processing` rather than
`container.additive` because this `Specimen`'s only `container` is the dispatched DNA tube
(the LSID), not the primary EDTA tube.

In [1]:
import pandas as pd

FIELD_MAPPING = [
    # csv_field, common_name, cardinality, field_type, hl7v2_field, fhir_field
    # cardinality/field_type source: RGL to SGL SOP v1.0 Appendix 3, in manifest column order.
    # hl7v2_field: transformation spec v0.5, plus the IG's own mappings where v0.5 has none.
    ("referral_id", "Original Order Placer Group Number", "MUST", "String",
     "ORC-4 Placer Group Number (ORC-4.2 = 8J834) and ORC-8 Parent (^r…&8J834, provisional). iGene ORM^O01 only: OBX-5 (OBX-3=NGIS_REFERRAL_ID), copied from ORC-4.1 by the RIE",
     "Parent ServiceRequest.identifier (FILL, intent original-order) and .subject (logical Group reference) + ServiceRequest.basedOn / requisition (NGISReferralId)"),
    ("clinical_indication_test_type_id", "Test Code", "OPTIONAL", "String",
     "OBR-4.1", "ServiceRequest.code.coding (England-GenomicTestDirectory)"),
    ("patient_nhs_number", "NHS Number", "REQUIRED IF AVAILABLE", "String",
     "PID-3 (NH)", "Patient.identifier (NHS number)"),
    ("patient_ngis_id", "Patient Identifier", "MUST", "String",
     "PID-3 (PID-3.4 = 8J834, PID-3.5 = PI - spec v0.5 had MR)",
     "Patient.identifier (PI, assigner Genomics England 8J834)"),
    ("patient_date_of_birth", "Date Of Birth", "MUST", "Date",
     "PID-7.1", "Patient.birthDate"),
    ("ordering_entity_id", "Original Ordering Facility Code", "OPTIONAL", "Code (ODS Code)",
     "ORC-21", "Specimen.identifier (as received) assigner"),
    ("glh_laboratory_id", "Filler Order Ordering Facility Code", "MUST", "Code (GEL GLH short code, e.g. now/yne)",
     "ORC-21 (ODS code via GEL_GLH_ODS)", "(ODS code via GEL_GLH_ODS) ServiceRequest.requester (PractitionerRole.organization when approved_by is present) / Specimen.identifier (LIMS) assigner"),
    ("primary_sample_received_date", "Sample Received Date", "OPTIONAL", "Date",
     "SPM-18 (primary SPM)", "Specimen.receivedTime (primary specimen)"),
    ("primary_sample_id_as_received_by_glh", "Received Sample Identifier", "OPTIONAL", "String",
     "- (not carried: SPM-2.1 is the LSID)", "Specimen.identifier:PlacerSpecimenNumber (PLAC, assigner ordering_entity_id)"),
    ("primary_sample_id_in_glh_lims", "LIMS Sample Identifier", "OPTIONAL", "String",
     "SPM-3 Specimen Parent IDs - Specimen Accession Number", "Specimen.accessionIdentifier (ACSN, assigner GLH)"),
    ("primary_sample_type", "Sample Type", "MUST", "Enum (e.g. Normal or Germline sample)",
     "SPM-11 (primary SPM, low confidence)", "Specimen.type / extension (germline vs tumour, low confidence)"),
    ("primary_sample_state", "Sample Material Type", "MUST", "Enum (e.g. blood_unsorted_edta)",
     "SPM-4.1 (primary SPM, SNOMED CT via GEL_SAMPLE_SOURCE_SNOMED)",
     "Split via GEL_SAMPLE_STATE: SampleSource -> Specimen.type (SNOMED CT + GEL coding, sampleState as text), Method -> Specimen.collection.method, StorageMedium -> Specimen.processing"),
    ("received_sample_topography", "Sample Topography", "OPTIONAL", "String",
     "SPM-8 (primary SPM, cancer only)", "Specimen.bodySite (cancer only)"),
    ("received_sample_morphology", "Sample Morphology", "OPTIONAL", "String", "", ""),
    ("received_sample_tumour_content_%", "Tumour Content", "OPTIONAL", "Number", "", ""),
    ("received_sample_comments", "Sample Comments", "OPTIONAL", "String", "", ""),
    ("received_sample_collection_date", "Specimen Collection Date", "OPTIONAL", "Date", "", ""),
    ("dispatched_sample_id_in_glh_lims", "Dispatched Sample Identifier", "OPTIONAL", "String",
     "", "Specimen.identifier:FillerSpecimenNumber (FILL, assigner GLH)"),
    ("dispatched_sample_lsid", "Specimen Barcode", "MUST", "String",
     "ORC-2.1/OBR-2.1 Placer Order Number (assigning authority = sending GLH ODS) + SPM-2.1. iGene ORM^O01 only: OBX-5 (OBX-3=DISPATCHED_SAMPLE_LSID), copied from SPM-2.1 by the RIE",
     "ServiceRequest.identifier:OrderIdentifier (PLAC, assigner = sending GLH ODS) + Specimen.container.identifier"),
    ("dispatched_sample_type", "Dispatched Sample Type", "MUST", "Enum (e.g. Normal or Germline sample)", "", ""),
    ("dispatched_sample_state", "Dispatched Material Type", "MUST", "Enum (GEL Product: DNA, RNA)",
     "SPM-4.1 (spec v0.5: RIE transforms DNA to iGene SampleType 'SAMPLE: Ext DNA' - not built in this worked example)",
     "Specimen.type (not built - single Specimen only carries primary_sample_state)"),
    ("dispatched_sample_volume_(ul)", "Sample Volume", "OPTIONAL", "Number",
     "SPM-12", "Specimen.collection.quantity"),
    ("laboratory_remaining_volume_banked_(ul)", "Remaining Banked Volume", "OPTIONAL", "Number", "", ""),
    ("glh_concentration_(ng/ul)", "DNA Concentration", "OPTIONAL", "Number", "", ""),
    ("glh_od260/280", "DNA Purity", "OPTIONAL", "Number", "", ""),
    ("glh_din_value", "DNA Integrity Number", "OPTIONAL", "Number", "", ""),
    ("glh_percentage_DNA_over_23kb", "DNA Fragment Size", "OPTIONAL", "Number", "", ""),
    ("glh_qc_status", "QC Status", "OPTIONAL", "String", "", ""),
    ("glh_sample_dispatch_date", "Dispatch Date", "OPTIONAL", "Date", "", ""),
    ("glh_sample_consignment_number", "Consignment Number", "OPTIONAL", "String",
     "SPM-32", "Specimen.identifier (type=STN)"),
    ("plating_organisation", "Plating Organisation", "OPTIONAL", "Enum", "", ""),
    ("gmc_rack_id", "Rack Identifier", "OPTIONAL", "String", "", ""),
    ("gmc_rack_well", "Rack Well Position", "OPTIONAL", "String (pattern)", "", ""),
    ("dna_extraction_protocol", "DNA Extraction Method", "OPTIONAL", "String",
     "", "Specimen.processing (DNA extraction step)"),
    ("prolonged_sample_storage", "Sample Storage Method", "OPTIONAL", "String", "", ""),
    ("retrospective_sample", "Retrospective Sample Flag", "OPTIONAL", "Enum",
     "(no clean v2 field)", "ServiceRequest.intent (reflex-order when Retrospective)"),
    ("approved_by", "Approved By", "OPTIONAL", "String", "",
     "ServiceRequest.requester -> PractitionerRole.practitioner.display (when present)"),
    ("patient_forename", "Forename", "MUST", "String", "PID-5.2", "Patient.name.given"),
    ("patient_surname", "Surname", "MUST", "String", "PID-5.1", "Patient.name.family"),
    ("family_structure", "Family Structure", "OPTIONAL", "Enumerated string",
     "OBX-5 (OBX-3=FAMILY_STRUCTURE)", "Observation (ObservationOrder, NWGMSA#FamilyStructure, subject the Patient) via the parent (original-order) ServiceRequest.supportingInfo"),
    ("participant_type", "Participant Type", "OPTIONAL", "Enumerated string",
     "OBX-5 (OBX-3=PARTICIPANT_TYPE)", "Observation (ObservationOrder, NWGMSA#ParticipantType) via the sub-order's ServiceRequest.supportingInfo"),
    ("clinical_information", "Clinical Information", "OPTIONAL", "String", "NTE-3", "ServiceRequest.note"),
]

field_mapping_df = pd.DataFrame(
    FIELD_MAPPING,
    columns=["csv_field", "common_name", "cardinality", "field_type", "hl7v2_field", "fhir_field"],
)

# Items 38-42: added to the national manifest for the NE&Y➜NW pathway only (SOP v1.0 Appendix 3).
NEY_NW_FIELDS = ["patient_forename", "patient_surname", "family_structure", "participant_type", "clinical_information"]
field_mapping_df["source"] = field_mapping_df.csv_field.map(
    lambda field: "national manifest (NE&Y➜NW item)" if field in NEY_NW_FIELDS else "national manifest")
manifest_fields = field_mapping_df.csv_field.tolist()
assert len(manifest_fields) == 42 and manifest_fields[37:] == NEY_NW_FIELDS
field_mapping_df

,csv_field,common_name,cardinality,field_type,hl7v2_field,fhir_field,source
0,referral_id,Original Order Placer Group Number,MUST,String,ORC-4 Placer Group Number (ORC-4.2 = 8J834) an...,"Parent ServiceRequest.identifier (FILL, intent...",national manifest
1,clinical_indication_test_type_id,Test Code,OPTIONAL,String,OBR-4.1,ServiceRequest.code.coding (England-GenomicTes...,national manifest
2,patient_nhs_number,NHS Number,REQUIRED IF AVAILABLE,String,PID-3 (NH),Patient.identifier (NHS number),national manifest
3,patient_ngis_id,Patient Identifier,MUST,String,"PID-3 (PID-3.4 = 8J834, PID-3.5 = PI - spec v0...","Patient.identifier (PI, assigner Genomics Engl...",national manifest
4,patient_date_of_birth,Date Of Birth,MUST,Date,PID-7.1,Patient.birthDate,national manifest
5,ordering_entity_id,Original Ordering Facility Code,OPTIONAL,Code (ODS Code),ORC-21,Specimen.identifier (as received) assigner,national manifest
6,glh_laboratory_id,Filler Order Ordering Facility Code,MUST,"Code (GEL GLH short code, e.g. now/yne)",ORC-21 (ODS code via GEL_GLH_ODS),(ODS code via GEL_GLH_ODS) ServiceRequest.requ...,national manifest
7,primary_sample_received_date,Sample Received Date,OPTIONAL,Date,SPM-18 (primary SPM),Specimen.receivedTime (primary specimen),national manifest
8,primary_sample_id_as_received_by_glh,Received Sample Identifier,OPTIONAL,String,- (not carried: SPM-2.1 is the LSID),Specimen.identifier:PlacerSpecimenNumber (PLAC...,national manifest
9,primary_sample_id_in_glh_lims,LIMS Sample Identifier,OPTIONAL,String,SPM-3 Specimen Parent IDs - Specimen Accession...,"Specimen.accessionIdentifier (ACSN, assigner GLH)",national manifest


## Sample manifest data

`Input/dWGS.csv` holds all 42 columns of the SOP v1.0 Digital Manifest. It's checked
against the mapping table above: every `MUST` field is populated, the type-sensitive fields
(`Number`, `Date`, `String (pattern)`) parse cleanly, and the optional enumerated NE&Y➜NW
fields hold a value the worked example knows how to code.

### The NE&Y➜NW fields are now in the manifest

Earlier passes of this notebook only had the 37 national columns. The patient's name came
from an NHS England PDS lookup by NHS number, and `family_structure`, `participant_type`
and `clinical_information` were labelled stand-ins. SOP v1.0 adds all five to the manifest
(items 38-42), and the transformation spec v0.5 takes PDS enrichment out of scope, so the
worked example now reads them straight from `Input/dWGS.csv`. The CSV keeps the same
values: names from `Input/PDS/*.csv` (and `EDITESTPATIENT` for the `9999999…` EDI test
patients, as in `Input/EDITestPatientsctDNA.csv`), and the former stand-ins for the family
and clinical fields. `patient_forename` and `patient_surname` are mandatory on this
pathway; the other three are optional.

The two other fields NW GMS asked for won't be added, and the spec v0.5 uses existing
fields in their place:

- **`placer_order_number`** - `dispatched_sample_lsid` is the interim Placer Order Number
  (`ORC-2.1`/`OBR-2.1`). `referral_id` and `patient_ngis_id` aren't unique per order (a
  family, or a cancer referral's germline and tumour samples, share them), but the LSID is
  unique per dispatched tube. See the ServiceRequest section.
- **`patient_medical_record_number`** - `patient_ngis_id` is the MRN-equivalent identifier
  (`PID-3` with type `MR`), which the spec is careful to say doesn't make it a hospital
  MRN. See the Patient section.

`family_structure` and `participant_type` are "enumerated strings" whose national
permitted values haven't been published yet (an open decision in spec v0.5). The values
here (`Singleton`/`Duo`/`Trio`, `Proband`/`Family Member`) follow the spec's own examples
and the IG's FamilyStructure value set; `Family Member` is this repo's placeholder.


In [2]:
dwgs_df = pd.read_csv("Input/dWGS.csv", dtype=str, keep_default_na=False)
print(f"{len(dwgs_df)} rows, {len(dwgs_df.columns)} columns")

assert list(dwgs_df.columns) == manifest_fields, \
    "Input/dWGS.csv must be exactly the 42 SOP v1.0 manifest columns, in the field mapping table's order"

# MUST fields must not be blank.
must_fields = field_mapping_df[field_mapping_df.cardinality == "MUST"].csv_field
for field in must_fields:
    blank_rows = dwgs_df.index[dwgs_df[field].str.strip() == ""].tolist()
    assert not blank_rows, f"{field} is MUST but blank in row(s) {blank_rows}"

# Number/Date fields must parse as their stated type.
for field in field_mapping_df[field_mapping_df.field_type == "Number"].csv_field:
    dwgs_df[field].map(lambda v: float(v) if v.strip() else None)
for field in field_mapping_df[field_mapping_df.field_type == "Date"].csv_field:
    pd.to_datetime(dwgs_df[field].replace("", pd.NaT))

# gmc_rack_well must match the SOP's stated pattern.
rack_well_pattern = r"^[A-H][0][1-9]|[A-H][1][0-2]$"
assert dwgs_df.gmc_rack_well.str.match(rack_well_pattern).all(), "gmc_rack_well values must match the SOP pattern"

# Enumerated values seen in a real GLH manifest export and the data mapping workbook.
SAMPLE_TYPES = {"Normal or Germline sample", "Solid tumour sample", "Liquid tumour sample"}
for field in ("primary_sample_type", "dispatched_sample_type"):
    assert dwgs_df[field].isin(SAMPLE_TYPES).all(), f"{field} must be one of {SAMPLE_TYPES}"
assert (dwgs_df.dispatched_sample_state == "DNA").all(), "dispatched_sample_state is expected to be DNA for dWGS"

# family_structure/participant_type are optional, but must be codeable when present.
assert dwgs_df.family_structure.isin({"", "Singleton", "Duo", "Trio"}).all()
assert dwgs_df.participant_type.isin({"", "Proband", "Family Member"}).all()

print("Input/dWGS.csv satisfies every cardinality/field_type constraint above.")
dwgs_df

11 rows, 42 columns
Input/dWGS.csv satisfies every cardinality/field_type constraint above.


,referral_id,clinical_indication_test_type_id,patient_nhs_number,patient_ngis_id,patient_date_of_birth,ordering_entity_id,glh_laboratory_id,primary_sample_received_date,primary_sample_id_as_received_by_glh,primary_sample_id_in_glh_lims,...,gmc_rack_well,dna_extraction_protocol,prolonged_sample_storage,retrospective_sample,approved_by,patient_forename,patient_surname,family_structure,participant_type,clinical_information
0,r26000000201,R14.1,9737383222,p26000000101,1978-01-17,RR8,yne,2026-08-20,S26-1A01,S26-1A01,...,A01,,,New,Duty Scientist on behalf of NEYGLH Medical,Rob,LEEDS,Singleton,Proband,Suspected inherited neurodevelopmental condition.
1,r26000000202,R59.1,9737873971,p26000000102,1947-04-27,RTR,yne,2026-08-20,S26-1A02,S26-1A02,...,A02,,,New,Duty Scientist on behalf of NEYGLH Medical,Allanys,MIDDLESBOROUGH,Duo,Proband,
2,r26000000202,R59.1,9999999603,p26000000103,1984-11-06,RTR,yne,2026-08-20,S26-1A03,S26-1A03,...,A03,,,New,Duty Scientist on behalf of NEYGLH Medical,Fourteen,EDITESTPATIENT,Duo,Family Member,
3,r26000000203,R27.3,9737383362,p26000000104,1989-07-01,RNN,yne,2026-08-20,S26-1A04,S26-1A04,...,A04,,,New,Duty Scientist on behalf of NEYGLH Medical,Gilly,BROUGH,Trio,Proband,Patient has intellectual disability
4,r26000000203,R27.3,9999999581,p26000000105,1960-01-01,RNN,yne,2026-08-20,S26-1A05,S26-1A05,...,A05,,,New,Duty Scientist on behalf of NEYGLH Medical,Thirteen,EDITESTPATIENT,Trio,Family Member,
5,r26000000203,R27.3,9999999514,p26000000106,1988-01-14,RNN,yne,2026-08-20,S26-1A06,S26-1A06,...,A06,,,New,Duty Scientist on behalf of NEYGLH Medical,Six,EDITESTPATIENT,Trio,Family Member,
6,r26000000204,R27.3,9737873866,p26000000107,1982-03-02,,yne,2026-09-28,S26-1A07,S26-1A07,...,A07,,,New,Duty Scientist on behalf of NEYGLH Medical,Euron,YORK,Singleton,Proband,
7,r26000000205,M27.16,9737873874,p26000000108,1965-03-22,,yne,2026-09-25,S26-1A08,S26-1A08,...,A08,,,New,Duty Scientist on behalf of NEYGLH Medical,Corlys,SUNDERLAND,Singleton,Proband,
8,r26000000205,M27.16,9737873874,p26000000108,1965-03-22,,yne,2026-09-25,S26-1A09,S26-1A09,...,A09,,,New,Duty Scientist on behalf of NEYGLH Medical,Corlys,SUNDERLAND,Singleton,Proband,
9,r26000000206,M91.1,9737873998,p26000000109,1993-12-27,,yne,2026-09-30,S26-1A10,S26-1A10,...,A10,,,New,Duty Scientist on behalf of NEYGLH Medical,Alicent,DURHAM,Singleton,Proband,


## Background

`Input/dWGS.csv` is a convenient stand-in, not a claim about the real world: in practice
each GLH/LIMS combination is likely to export a different local format, and this
notebook's manifest is only one plausible shape used here to get test data flowing
end-to-end. The field mapping above is the part expected to generalise; the CSV's exact
column names are not.

### Where this sits: a sub-contracted order

At a high level, North East and Yorkshire (NE&Y) Genomics referring a sample on to North
West Genomics for sequencing is a **sub-contracted order**. In NW-GMSA's own
[Inter-Laboratory Workflow (ILW) page](https://nw-gmsa.github.io/en/ILW.html), this is
**`LAB-35`** (Sub-order Management) — a laboratory (NE&Y Genomics, acting as an *Order
Placer* toward NW Genomics) sending a sub-order to another laboratory (NW Genomics,
acting as *Order Filler*) for testing it can't do itself, distinct from `LAB-1` (the
original clinical order a referring Trust placed with NE&Y Genomics in the first place).
Results come back the same way, as `LAB-36`.

Per [NW-GMSA's Message Exchange page](https://nw-gmsa.github.io/en/MQ.html), a `LAB-35`
order like this can be sent as **either** a FHIR `Bundle` (`POST [base]/$process-message`,
the [laboratory-order `MessageDefinition`](MessageDefinition-laboratory-order.html)) or
HL7 v2 `OML^O21` (MLLP, [documented here](hl7v2.html#oml_o21-laboratory-order)) — the
same choice `03-laboratory-order-from-csv.ipynb` exercised for a different kind of
order. Both follow the *same* underlying data model (next section), so the choice is
purely about what the sending system can produce.

### Why the format actually sent will vary

Which format NE&Y Genomics uses will depend on how each constituent lab implements it.
"NE&Y Genomics" isn't one building — the
[Yorkshire and North East GLH](https://directory.spineservices.nhs.uk/ORD/2-0-0/organisations/699N0)
is a partnership of laboratories. The manifest's `glh_laboratory_id` only identifies the
GLH as a whole (`yne`), not which of these constituent labs handled the sample. Their ODS
codes, confirmed against the live
[NHS ODS API](https://directory.spineservices.nhs.uk/ORD/2-0-0/organisations), are:

- **Northern Genetics Service**, Newcastle — `699N0` (the GLH's own registration,
  based at Newcastle's Institute for Human Genetics - not `RTD07`, that site's separate
  NHS Trust Site identity, which this manifest field doesn't use)
- **Sheffield Diagnostic Genetics Service** — `699D0`
- **The Leeds Genetics Laboratory** — `699C0`

Some of these may already run a LIMS capable of sending `ORM^O01`/`OML^O21` (or FHIR)
directly; others may only export a flat file, needing a data pipeline in between to
produce a proper order message. `Input/dWGS.csv` is standing in for that second case.

![notebook 8 diagram 3](https://mermaid.ink/svg/Zmxvd2NoYXJ0IFRCCiAgICBzdWJncmFwaCBHTEhbIk5FJlkgR2Vub21pY3MiXQogICAgICAgIE5DTFsiTm9ydGhlcm4gR2VuZXRpY3MgU2VydmljZTxici8+TmV3Y2FzdGxlIC0gNjk5TjAiXQogICAgICAgIFNIRlsiU2hlZmZpZWxkIERpYWdub3N0aWM8YnIvPkdlbmV0aWNzIFNlcnZpY2UgLSA2OTlEMCJdCiAgICAgICAgTERTWyJUaGUgTGVlZHMgR2VuZXRpY3M8YnIvPkxhYm9yYXRvcnkgLSA2OTlDMCJdCiAgICBlbmQKICAgIFBJUEVbIk1hbmlmZXN0IENTViAtPiBtZXNzYWdlPGJyLz5kYXRhIHBpcGVsaW5lPGJyLz4odGhpcyBub3RlYm9vaydzIHdvcmtlZCBleGFtcGxlKSJdCiAgICBSSUVbIk5XIEdlbm9taWNzPGJyLz5SZWdpb25hbCBJbnRlZ3JhdGlvbiBFbmdpbmUgKFJJRSk8YnIvPjY5OVgwIl0KCiAgICBOQ0wgLS0gIkxJTVMgc2VuZHMgT01MXk8yMSAvIE8yMSBGSElSIE1lc3NhZ2UgZGlyZWN0bHkiIC0tPiBSSUUKICAgIExEUyAtLSAiTElNUyBzZW5kcyBPTUxeTzIxIC8gTzIxIEZISVIgTWVzc2FnZSBkaXJlY3RseSIgLS0+IFJJRQogICAgU0hGIC0tICJMSU1TIGV4cG9ydHMgYSBtYW5pZmVzdCBDU1YiIC0tPiBQSVBFCiAgICBQSVBFIC0tICJPTUxeTzIxIC8gTzIxIEZISVIgTWVzc2FnZSIgLS0+IFJJRQ==)

<!--
```mermaid
flowchart TB
    subgraph GLH["NE&Y Genomics"]
        NCL["Northern Genetics Service<br/>Newcastle - 699N0"]
        SHF["Sheffield Diagnostic<br/>Genetics Service - 699D0"]
        LDS["The Leeds Genetics<br/>Laboratory - 699C0"]
    end
    PIPE["Manifest CSV -> message<br/>data pipeline<br/>(this notebook's worked example)"]
    RIE["NW Genomics<br/>Regional Integration Engine (RIE)<br/>699X0"]

    NCL -- "LIMS sends OML^O21 / O21 FHIR Message directly" --> RIE
    LDS -- "LIMS sends OML^O21 / O21 FHIR Message directly" --> RIE
    SHF -- "LIMS exports a manifest CSV" --> PIPE
    PIPE -- "OML^O21 / O21 FHIR Message" --> RIE
```
-->

### The shared data model

Both the v2 and FHIR `O21` orders follow the same model, defined once in the NW
Implementation Guide and rendered as whichever wire format the sender chooses:

![notebook 8 diagram 4](https://mermaid.ink/svg/ZXJEaWFncmFtCiAgICBNZXNzYWdlSGVhZGVyIHx8LS18fCBTZXJ2aWNlUmVxdWVzdCA6IGZvY3VzCiAgICBQYXRpZW50IHx8LS1veyBTZXJ2aWNlUmVxdWVzdCA6IHN1YmplY3QKICAgIFNlcnZpY2VSZXF1ZXN0IH1vLS18fCBQYXJlbnRTZXJ2aWNlUmVxdWVzdCA6IGJhc2VkT24KICAgIFBhcmVudFNlcnZpY2VSZXF1ZXN0IH1vLS18fCBHcm91cCA6ICJzdWJqZWN0IChyZWZlcnJhbF9pZCkiCiAgICBTZXJ2aWNlUmVxdWVzdCB8fC0tb3sgU3BlY2ltZW4gOiBzcGVjaW1lbgogICAgU2VydmljZVJlcXVlc3QgfHwtLW98IFBhcnRpY2lwYW50VHlwZU9ic2VydmF0aW9uIDogc3VwcG9ydGluZ0luZm8=)

<!--
```mermaid
erDiagram
    MessageHeader ||--|| ServiceRequest : focus
    Patient ||--o{ ServiceRequest : subject
    ServiceRequest }o--|| ParentServiceRequest : basedOn
    ParentServiceRequest }o--|| Group : "subject (referral_id)"
    ServiceRequest ||--o{ Specimen : specimen
    ServiceRequest ||--o| ParticipantTypeObservation : supportingInfo
```
-->

- **`MessageHeader`** (`MSH`)
- **`Patient`** (`PID`)
- **`ServiceRequest`** (`ORC` + `OBR`) - the `LAB-35` sub-order (`intent` `filler-order`)
- **Parent `ServiceRequest`** - the NGIS referral the sub-order is based on (`intent`
  `original-order`), identified by `referral_id` as its Order Filler Number. Its
  `subject` is the referral's family, a logical reference to a `Group` with the same
  `referral_id` as its identifier (the `Group` itself isn't sent). Every family member's
  sub-order is based on a parent with the same `referral_id`, and carries it as its
  `requisition` (Order Group Number, `ORC-4`), which is what ties a Duo or Trio
  together. Each family member is still ordered in a separate message.
- **`Specimen`** (`SPM`)
- **`Observation`** ([`ObservationOrder`](https://nw-gmsa.github.io/en/StructureDefinition-ObservationOrder.html),
  `OBX`) - the ask-at-order-entry answer for this patient's role in the family
  (`participant_type`), from the sub-order's `supportingInfo`. An earlier pass of this
  notebook modelled `participant_type` and `family_structure` as a `FamilyGeneticGroup`
  `Group` listing the whole family.

**NW Genomics' own LIMS speaks HL7 v2 internally, always** — regardless of which format
this sub-order arrives in, the RIE is what transforms it (FHIR or v2) into NW Genomics'
internal v2 shape before it ever reaches the LIMS. The worked example below builds the
external-facing FHIR order and converts it to v2 the same way the RIE's own
`transformToV2` tooling does — it does not reach into the LIMS's internal format.

### Room to extend beyond what's specified

Although the NW data model above names specific FHIR and v2 elements, it doesn't forbid
others - both standards still accept elements it doesn't mention, as long as they're
individually conformant to the schema (and, on the FHIR side, pass validation - this
repo's own RIE converts inbound v2 to FHIR and validates that internally too, the same
check `03-laboratory-order-from-csv.ipynb`/`04-laboratory-report-fhir-from-hl7v2.ipynb`
run explicitly). So NE&Y Genomics is free to add fields of its own - exactly how items 38-42 in the
field-mapping table above started out, before NHS England adopted them into the SOP - as
long as they stay conformant to both schemas.

This is the main advantage either HL7 standard has over exchanging flat files like
`Input/dWGS.csv` directly: producer and consumer aren't tightly coupled to one exact,
frozen column layout the way a raw CSV exchange would be. A new field can be added by
one side without breaking the other, which is what actually makes a CSV-based interface
brittle and costly to change compared to a schema built to expect extension.

(A worked example of the reverse direction - HL7 v2 into FHIR - is
`04-laboratory-report-fhir-from-hl7v2.ipynb`, which converts a `ORU^R01` laboratory
report the same hand-built way this notebook builds an order.)

## FHIR worked example

Same approach as `03-laboratory-order-from-csv.ipynb`: build the order by hand in
Python, validate resource-by-resource, then assemble and validate the whole `Bundle`.
Row 0 of `Input/dWGS.csv` (referral `r26000000201`, Rob Leeds, NHS number `9737383222` -
the same patient used throughout `03`-`05`) is the source data.

To keep this tractable, the worked example only builds the fields that already have a
confirmed (non-blank) `hl7v2_field`/`fhir_field` in the mapping table above. Every
`OPTIONAL` field still mapped as `""`/`""` (topography, morphology, QC metrics, rack
position, and so on) is a genuine open question for a future pass, not an oversight -
see the closing note.

In [3]:
import json
import os
import subprocess
import tempfile
from datetime import datetime
from uuid import uuid4

import requests
from dotenv import load_dotenv

load_dotenv()

toolsServer = os.getenv("V2_TOOLS")

NHS_NUMBER_SYSTEM = "https://fhir.nhs.uk/Id/nhs-number"
ODS_SYSTEM = "https://fhir.nhs.uk/Id/ods-organization-code"
V2_0203 = "http://terminology.hl7.org/CodeSystem/v2-0203"
SNOMED_SYSTEM = "http://snomed.info/sct"
GENOMIC_TEST_DIRECTORY_SYSTEM = "https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirectory"
NW_GLH_ODS = "699X0"        # NW Genomics - the LAB-35 sub-order's Order Filler
GENOMICS_ENGLAND_ODS = "8J834"  # Genomics England - runs NGIS (National Genomic Informatics
                                 # System), the assigner of patient_ngis_id below

# glh_laboratory_id is Genomics England's short GLH code, not an ODS code - this maps it to
# the GLH's ODS code wherever FHIR/v2 needs one. "now" is from a real NW GLH manifest export;
# "yne" is assumed for NE&Y and not yet confirmed against a real NE&Y export.
GEL_GLH_ODS = {
    "yne": "699N0",  # YORKSHIRE AND NORTH EAST GLH
    "now": NW_GLH_ODS,
}

# primary_sample_state is GEL's TOMS sampleState (GelReportModels participant 1.3.0,
# GermlineSample/TumourSample.sampleState: "the sampleState as entered by the clinician in
# TOMs") - free text, not an enumeration. Its values combine three GEL enumerations, so it's
# split into them: SampleSource -> Specimen.type, Method -> Specimen.collection.method and
# StorageMedium -> Specimen.processing. The split is inferred from the values seen so far -
# GEL doesn't publish the full sampleState list. GEL publishes no FHIR CodeSystems either, so
# these system URIs are provisional.
GEL_SAMPLE_SOURCE_SYSTEM = "https://gelreportmodels.genomicsengland.co.uk/CodeSystem/SampleSource"
GEL_METHOD_SYSTEM = "https://gelreportmodels.genomicsengland.co.uk/CodeSystem/Method"
GEL_STORAGE_MEDIUM_SYSTEM = "https://gelreportmodels.genomicsengland.co.uk/CodeSystem/StorageMedium"
GEL_SAMPLE_STATE = {
    # sampleState: (SampleSource, Method, StorageMedium)
    "blood_unsorted_edta": ("BLOOD", "UNSORTED", "EDTA"),
    "boneMarrow_unsorted_edta": ("BONE_MARROW", "UNSORTED", "EDTA"),
    "tumour_freshFrozen": ("TUMOUR", None, "FF"),  # no Method part
}
# Specimen.type also needs a SNOMED CT code from NW-GMSA's specimen-type ValueSet
# (https://nw-gmsa.github.io/en/ValueSet-specimen-type.html) - the profile's binding is required.
GEL_SAMPLE_SOURCE_SNOMED = {
    "BLOOD": ("258580003", "Whole blood specimen (specimen)"),
    "BONE_MARROW": ("119359002", "Bone marrow specimen (specimen)"),
    "TUMOUR": ("441479001", "Fresh tissue specimen (specimen)"),  # nearest code in the ValueSet
}

# NE&Y Genomics' three constituent labs' ODS codes, confirmed against the live NHS ODS API (see
# Background above). 699N0 is also the GLH's own registration, so it's what GEL_GLH_ODS["yne"] gives.
NEY_GLH_NAME = {
    "699N0": "Northern Genetics Service, Newcastle (NE&Y Genomics)",
    "699D0": "Sheffield Diagnostic Genetics Service (NE&Y Genomics)",
    "699C0": "The Leeds Genetics Laboratory (NE&Y Genomics)",
}

NWGMSA_PATIENT_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/Patient"
NWGMSA_SERVICE_REQUEST_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/ServiceRequest"
NWGMSA_SPECIMEN_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/Specimen"
NWGMSA_OBSERVATION_ORDER_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/ObservationOrder"
NWGMSA_PRACTITIONER_ROLE_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/PractitionerRole"
NWGMSA_CODESYSTEM = "https://fhir.nwgenomics.nhs.uk/CodeSystem/NWGMSA"
NGIS_REFERRAL_ID_SYSTEM = "https://genomicsengland.co.uk/healthcare/referral-id"
NGIS_PARTICIPANT_ID_SYSTEM = "https://genomicsengland.co.uk/healthcare/participant-id"
NGIS_LAB_SAMPLE_ID_SYSTEM = "https://genomicsengland.co.uk/healthcare/lab-sample-id"
NGIS_PRIMARY_SAMPLE_ID_SYSTEM = "https://genomicsengland.co.uk/healthcare/primary-sample-id"
# Identifier profiles (not declarable on the identifier itself - documented for traceability)
NGIS_REFERRAL_ID_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/NGISReferralId"
ORDER_IDENTIFIER_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/OrderIdentifier"


def details(item):
    return item["text"]


def issues_df(outcome):
    df = pd.DataFrame(outcome["issue"])
    df["details"] = df["details"].apply(details)
    df.drop(columns=["extension"], inplace=True, errors="ignore")
    df.sort_values(by=["severity"], inplace=True)
    df = df[~df["details"].str.contains("ValueSet/mimetypes")]
    df = df[~df["details"].str.contains("failed: dom-6")]
    df = df[~df["details"].str.contains("bcp:13")]
    df = df[~df["details"].str.contains("terminology service")]
    return df


def validate_resource(resource, profile):
    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as tmp:
        json.dump(resource, tmp)
        tmp_path = tmp.name
    outcome_path = tmp_path + "-OperationOutcome.json"
    subprocess.run(
        [
            "java", "-jar", "validator_cli.jar", tmp_path,
            "-version", "4.0.1", "-ig", "package.tgz",
            "-profile", profile, "-tx", "n/a",
            "-output", outcome_path, "-output-style", "json",
        ],
        capture_output=True,
    )
    with open(outcome_path) as f:
        return issues_df(json.load(f))


row = dwgs_df.iloc[0]
glh_ods = GEL_GLH_ODS[row["glh_laboratory_id"]]
row[["referral_id", "clinical_indication_test_type_id", "patient_nhs_number", "patient_ngis_id",
     "ordering_entity_id", "glh_laboratory_id", "primary_sample_id_as_received_by_glh",
     "dispatched_sample_lsid", "glh_sample_consignment_number"]]

referral_id                                   r26000000201
clinical_indication_test_type_id                     R14.1
patient_nhs_number                              9737383222
patient_ngis_id                               p26000000101
ordering_entity_id                                     RR8
glh_laboratory_id                                      yne
primary_sample_id_as_received_by_glh              S26-1A01
dispatched_sample_lsid                          4000000001
glh_sample_consignment_number           NEY GLH 2026-08-26
Name: 0, dtype: str

### NE&Y➜NW fields: name, family and clinical information

As described under [Sample manifest data](#Sample-manifest-data), the patient's name and
the family and clinical fields are items 38-42 of the SOP v1.0 manifest, so they come
straight from `row`. `family_structure`, `participant_type` and `clinical_information`
are optional on this pathway, so each is only built into the order when it's present.


In [4]:
patient_forename = row["patient_forename"]
patient_surname = row["patient_surname"]

print(f"{patient_forename} {patient_surname}")
row[["family_structure", "participant_type", "clinical_information"]]


Rob LEEDS


family_structure                                                Singleton
participant_type                                                  Proband
clinical_information    Suspected inherited neurodevelopmental condition.
Name: 0, dtype: str

### Patient

`name` comes from the manifest's `patient_forename`/`patient_surname` (SOP v1.0 items
38-39).

`patient_ngis_id` is a patient identifier, not a hospital MRN, so it's coded `PI`, and is
sent as `PID-3` type `PI`, including to iGene. The transformation spec v0.5 sends it as
the MRN-equivalent `MR`, because iGene's existing `patientID` mapping reads the MR
repetition; whether iGene accepts `PI` is being tested. (An earlier pass followed the spec
and coded it `MR`.) The referring Trust's own MRN still isn't in the manifest.

`patient_ngis_id` has no confirmed FHIR identifier *system* URI, but it does have a
confirmed **assigner**: NGIS (the National Genomic Informatics System) is a service run
by Genomics England, whose ODS code is `8J834` (confirmed against the live
[NHS ODS API](https://directory.spineservices.nhs.uk/ORD/2-0-0/organisations/8J834)) -
so rather than guess at a `system` URI, the identifier below carries `assigner` instead,
the same ODS-organisation-code pattern `Specimen.identifier.assigner` uses further down.
This resolves the field-mapping table's earlier `(TBC system)` flag for this field.

In [5]:
patient_fullurl = f"urn:uuid:{uuid4()}"
patient = {
    "resourceType": "Patient",
    "identifier": [
        {"system": NHS_NUMBER_SYSTEM, "type": {"coding": [{"system": V2_0203, "code": "NH"}]}, "value": row["patient_nhs_number"]},
        {"system": NGIS_PARTICIPANT_ID_SYSTEM, "assigner": {"identifier": {"system": ODS_SYSTEM, "value": GENOMICS_ENGLAND_ODS}},
         "type": {"coding": [{"system": V2_0203, "code": "PI"}]}, "value": row["patient_ngis_id"]},  # NGIS Patient ID (spec v0.5 has MR)
    ],
    "name": [{"family": patient_surname, "given": [patient_forename]}],  # SOP v1.0 items 38-39
    "birthDate": row["patient_date_of_birth"],
}
print(json.dumps(patient, indent=2))

{
  "resourceType": "Patient",
  "identifier": [
    {
      "system": "https://fhir.nhs.uk/Id/nhs-number",
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "NH"
          }
        ]
      },
      "value": "9737383222"
    },
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "8J834"
        }
      },
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "PI"
          }
        ]
      },
      "value": "p26000000101"
    }
  ],
  "name": [
    {
      "family": "LEEDS",
      "given": [
        "Rob"
      ]
    }
  ],
  "birthDate": "1978-01-17"
}


In [6]:
validate_resource(patient, NWGMSA_PATIENT_PROFILE)

,severity,code,details,expression


### Parent ServiceRequest and ask-at-order-entry Observations

The NGIS referral (`referral_id`) is the **original order** this sub-order is based on, so
it's modelled as a parent `ServiceRequest`:

- **`intent`** - `original-order`.
- **`identifier`** - `referral_id` as an **Order Filler Number**: `type` v2-0203 `FILL`,
  system `NGIS-ReferralIdentifier`, assigned by Genomics England (`8J834`, the same NGIS
  assigner as `patient_ngis_id` above), which issues it from TOMS.
- **`code`** - the requested test.
- **`subject`** - the referral's family: a **logical reference** to a `Group`, with
  `referral_id` as its identifier (the IG's `NGISReferralId`: system
  `NGIS-ReferralIdentifier`, `type` `PGN`, assigner `8J834`). No `Group` resource is sent.
  The IG's `ServiceRequest` profile allows a `FamilyGeneticGroup` subject for the dWGS
  original order only.
- **`requester`** - the referring Trust (`ordering_entity_id`) that placed the original
  order, or the GLH when that's blank.

This follows the IG's existing parent-order pattern (`Bundle-Message-FillerOrder`: an
`original-order` with a `FILL` identifier, and a child order whose `basedOn` references it
and whose `requisition` carries the same value). The sub-order below does the same.

`family_structure` and `participant_type` are ask-at-order-entry answers, each an
[`ObservationOrder`](https://nw-gmsa.github.io/en/StructureDefinition-ObservationOrder.html)
`Observation` (the `OrderEntryQuestions` slice of `supportingInfo`) coded from NW-GMSA's
`NWGMSA` CodeSystem (the same codes as the `NOS/FamilyStructure`/`NOS/ParticipantType`
Questionnaire items), on the order each one describes. Both are about the `Patient`: the
IG's `ObservationOrder` profile only allows a `Patient` subject, not the family `Group`.

- **`family_structure`** describes the referral, not one person, so its `Observation` is
  referenced from the **parent's** `supportingInfo`: `code` `FamilyStructure`, value
  `FamilyStructureSingleton`, `FamilyStructureDuo` or `FamilyStructureTrio`.
- **`participant_type`** describes this participant, so its `Observation` is referenced
  from the **sub-order's** `supportingInfo`: `code` `ParticipantType`, value
  `WGSParticipantTypeProband` or `WGSParticipantTypeFamilyMember`.

Both are optional on this pathway, so each `Observation` is only built when the manifest
supplies a value. The family structure isn't inferred from the messages (or from how many
manifest rows share a `referral_id`). In v2 the RIE sends only the sub-order (the filler
order), so both become `OBX`s under it (`FAMILY_STRUCTURE`, `PARTICIPANT_TYPE`). Each
message doesn't list the other family members either:
a family is the set of sub-orders based on a parent with the same `referral_id`. How a
family member is related to the proband (mother, father, sibling...) still isn't in the
manifest.


In [7]:
# NWGMSA CodeSystem answers for the dWGS ask-at-order-entry questions.
FAMILY_STRUCTURE_CODES = {
    "Singleton": "FamilyStructureSingleton",
    "Duo": "FamilyStructureDuo",
    "Trio": "FamilyStructureTrio",
}
PARTICIPANT_TYPE_CODES = {
    "Proband": "WGSParticipantTypeProband",
    "Family Member": "WGSParticipantTypeFamilyMember",
}


def order_observation(subject, code, display, value_code, value_display):
    """ObservationOrder - an ask-at-order-entry answer, referenced from ServiceRequest.supportingInfo."""
    return {
        "resourceType": "Observation",
        "meta": {"profile": [NWGMSA_OBSERVATION_ORDER_PROFILE]},
        "identifier": [{"system": "urn:ietf:rfc:3986", "value": f"urn:uuid:{uuid4()}"}],
        "status": "final",
        "code": {"coding": [{"system": NWGMSA_CODESYSTEM, "code": code, "display": display}]},
        "subject": subject,
        "valueCodeableConcept": {"coding": [{"system": NWGMSA_CODESYSTEM, "code": value_code, "display": value_display}]},
    }


# The NGIS Referral Id, assigned by Genomics England (TOMS). It identifies the referral's family
# (the parent's subject Group), is the original order's Order Filler Number (FILL), and the
# sub-order's requisition (Order Group Number, PGN) below.
ngis_referral_id = {
    "system": NGIS_REFERRAL_ID_SYSTEM,
    "value": row["referral_id"],
    "assigner": {"identifier": {"system": ODS_SYSTEM, "value": GENOMICS_ENGLAND_ODS}},
}
referral_filler_number = {**ngis_referral_id, "type": {"coding": [{"system": V2_0203, "code": "FILL"}]}}
referral_identifier = {**ngis_referral_id, "type": {"coding": [{"system": V2_0203, "code": "PGN"}]}}

parent_service_request_fullurl = f"urn:uuid:{uuid4()}"
parent_service_request = {
    "resourceType": "ServiceRequest",
    "status": "active",
    "intent": "original-order",
    "category": [{"coding": [{"system": "http://snomed.info/sct", "code": "116148004"}]}],
    "code": {"coding": [{"system": GENOMIC_TEST_DIRECTORY_SYSTEM, "code": row["clinical_indication_test_type_id"]}]},
    "identifier": [referral_filler_number],
    "subject": {"type": "Group", "identifier": referral_identifier},  # logical reference - the referral's family (NGISReferralId)
    # The referring Trust placed the original order; fall back to the GLH when it isn't supplied.
    "requester": {"type": "Organization",
                  "identifier": {"system": ODS_SYSTEM, "value": row["ordering_entity_id"] or glh_ods}},
}

# Optional on the NE&Y➜NW pathway - only built when the manifest supplies a value.
observations = {}  # fullUrl -> Observation
# family_structure describes the referral, so it goes on the parent. ObservationOrder only allows a
# Patient subject, so it's about this participant rather than the family Group.
if row["family_structure"]:
    family_structure_fullurl = f"urn:uuid:{uuid4()}"
    observations[family_structure_fullurl] = order_observation(
        {"reference": patient_fullurl}, "FamilyStructure", "Family Structure",
        FAMILY_STRUCTURE_CODES[row["family_structure"]], row["family_structure"])
    parent_service_request["supportingInfo"] = [{"reference": family_structure_fullurl, "type": "Observation"}]
# participant_type describes this participant, so it goes on the sub-order.
participant_type_fullurl = None
if row["participant_type"]:
    participant_type_fullurl = f"urn:uuid:{uuid4()}"
    observations[participant_type_fullurl] = order_observation(
        {"reference": patient_fullurl}, "ParticipantType", "Participant Type",
        PARTICIPANT_TYPE_CODES[row["participant_type"]], row["participant_type"])

print(json.dumps(parent_service_request, indent=2))
for observation in observations.values():
    print(json.dumps(observation, indent=2))


{
  "resourceType": "ServiceRequest",
  "status": "active",
  "intent": "original-order",
  "category": [
    {
      "coding": [
        {
          "system": "http://snomed.info/sct",
          "code": "116148004"
        }
      ]
    }
  ],
  "code": {
    "coding": [
      {
        "system": "https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirectory",
        "code": "R14.1"
      }
    ]
  },
  "identifier": [
    {
      "system": "https://genomicsengland.co.uk/healthcare/referral-id",
      "value": "r26000000201",
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "8J834"
        }
      },
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "FILL"
          }
        ]
      }
    }
  ],
  "subject": {
    "type": "Group",
    "identifier": {
      "system": "https://genomicsengland.co.uk/healthcare/referral-

In [8]:
pd.concat([
    validate_resource(parent_service_request, NWGMSA_SERVICE_REQUEST_PROFILE).assign(resource="parent ServiceRequest"),
    *[validate_resource(o, NWGMSA_OBSERVATION_ORDER_PROFILE).assign(resource=o["code"]["coding"][0]["code"])
      for o in observations.values()],
])


,severity,code,details,expression,resource
0,information,informational,This element does not match any known slice de...,[ServiceRequest.supportingInfo[0]],parent ServiceRequest
1,warning,invalid,"Best Practice Recommendation: In general, all ...",[Observation],FamilyStructure
2,warning,invalid,"Best Practice Recommendation: In general, all ...",[Observation],FamilyStructure
1,warning,invalid,"Best Practice Recommendation: In general, all ...",[Observation],ParticipantType
2,warning,invalid,"Best Practice Recommendation: In general, all ...",[Observation],ParticipantType


The parent `ServiceRequest` validates cleanly against IG 2.2.6, including its `subject`:
the IG's `ServiceRequest` profile now allows a `FamilyGeneticGroup` subject identified by
an `NGISReferralId` for the dWGS original order. (Against IG 2.2.1 the same `subject` gave
two errors, because that profile only allowed a `Patient` subject with an NHS number.)
Everything else about
the parent (`intent`, the `FILL` Order Filler Number, `requester`) passes. The
`ParticipantType` `Observation` passes the `ObservationOrder` profile; its two warnings are
best-practice recommendations (no `performer` or `effective[x]`, neither of which the
manifest supplies). The earlier `FamilyGeneticGroup` pass failed instead on the `PGN`
identifier type, against the validator's older offline copy of `v2-0203`.


### Specimen

One `Specimen`, not two: `dWGS.csv`'s primary-specimen and dispatched-specimen fields
both describe the same physical sample as it moves through this sub-order, so they're
carried as multiple identifiers on a single resource rather than two linked resources.
The identifiers have different **assigners**, reflecting who actually issued each one:

- **Received Sample Identifier** (`primary_sample_id_as_received_by_glh`) - assigned by
  `ordering_entity_id` (Leeds, `RR8`), the *original* order placer who received and
  labelled the specimen before it was sent on. Coded `type` `PLAC` ("Placer
  Identifier") - the identifier assigned by whoever placed the order/collected the
  specimen.
- **LIMS Sample Identifier** (`primary_sample_id_in_glh_lims`) - NE&Y Genomics' own LIMS
  number for the specimen once it was logged there, assigned by `glh_laboratory_id`
  (`yne`, so ODS `699N0`). That's a laboratory's own specimen number, so it is
  `Specimen.accessionIdentifier` (the IG's
  [Specimen Accession Number](https://nw-gmsa.github.io/en/StructureDefinition-SpecimenAccessionNumber.html),
  `type` `ACSN`), which the IG maps to `SPM-3` (Specimen Parent IDs).
- **Dispatched Sample Identifier** (`dispatched_sample_id_in_glh_lims`) - the extracted
  DNA's number in NE&Y Genomics' LIMS, also assigned by `glh_laboratory_id`. Coded `type`
  `FILL` ("Filler Identifier") - the `FillerSpecimenNumber` slice. Built only when present
  (it's optional nationally).

`glh_sample_consignment_number` becomes the `identifier:ShipmentTrackingNumber` slice,
coded `type` `STN` ("Shipment Tracking Number"), per the field-mapping table above.
Those `type` codes (and `ACSN`) are from
[HL7's v2-0203 CodeSystem](http://terminology.hl7.org/CodeSystem/v2-0203) - the same
system already used for `Patient.identifier.type` above - rather than left text-only.

`dispatched_sample_lsid` (Specimen Barcode) becomes `Specimen.container.identifier`,
coded `type` `ZCID`, and is the only value in `SPM-2` (`SPM-2.1`) - unlike
the other three, `ZCID` is a NE&Y Genomics locally-defined ("Z") identifier-type code,
not one of HL7's own v2-0203 values, so it's coded against the NW-GMSA `NWGMSA`
CodeSystem (`https://fhir.nwgenomics.nhs.uk/CodeSystem/NWGMSA`, in the IG's
`NWIdentifierType` ValueSet) rather than v2-0203, in the same `type.coding` shape.

`primary_sample_state` no longer stays text-only either. `Input/dWGS.csv` carries GEL's
TOMS `sampleState` (e.g. `blood_unsorted_edta`), the same as a real GLH manifest export,
and `GEL_SAMPLE_STATE` splits it into its three GEL enumerations (see the field mapping
section above): `SampleSource` (`BLOOD`) becomes `Specimen.type`, coded both in GEL and as
a SNOMED CT code from NW-GMSA's own [specimen-type
ValueSet](https://nw-gmsa.github.io/en/ValueSet-specimen-type.html) (e.g. `258580003`
Whole blood specimen), with the original value as `type.text`; `Method` (`UNSORTED`)
becomes `collection.method`; and `StorageMedium` (`EDTA`) becomes a `processing` step. `primary_sample_type` (the germline/tumour classifier)
and `dispatched_sample_state` remain text-only/unused, though - `primary_sample_type`
is still flagged low-confidence in the field-mapping table above, and nothing in this
worked example builds `dispatched_sample_state` into any resource yet. It's the GEL
`Product` code (`DNA`), but this single `Specimen`'s `type` describes the primary
material, so `DNA` needs the separate dispatched `Specimen` (with `Specimen.parent`)
described in the field mapping section above (see the discrepancy check below). For
iGene, the transformation spec v0.5 has the RIE turn `DNA` into the iGene SampleType
`SAMPLE: Ext DNA` in `SPM-4.1` - a value transform inside NW Genomics, downstream of this
external order.

Three more fields land on `Specimen.processing`/`Specimen.collection`:
`dna_extraction_protocol` becomes a `processing` step ("DNA extraction") when present -
text-only, since its values (e.g. `"Qiagen"`) name a vendor/kit, not a coded procedure
with a confirmed SNOMED CT equivalent. It's blank in `Input/dWGS.csv`, as in the real
export: the data mapping workbook says covaris/qiagen only apply to FFPE samples, and
these are germline blood. It used to go in `collection.method`, but that's *how the
specimen was collected*, which the GEL `Method` part of `primary_sample_state` now
supplies. `dispatched_sample_volume_(ul)` becomes
`collection.quantity`, a proper UCUM `Quantity` (`uL`) rather than free text.
`received_sample_topography` would become `bodySite` - `MUST (cancer only)`, and
`Input/dWGS.csv`'s rare-disease rows correctly leave it blank, so it's built only when
present rather than added as an empty `CodeableConcept`.

In [9]:
specimen_fullurl = f"urn:uuid:{uuid4()}"
sample_source, sample_method, storage_medium = GEL_SAMPLE_STATE[row["primary_sample_state"]]
primary_sample_type_code, primary_sample_type_display = GEL_SAMPLE_SOURCE_SNOMED[sample_source]
specimen = {
    "resourceType": "Specimen",
    "identifier": [
        {"type": {"coding": [{"system": V2_0203, "code": "PLAC"}]}, "value": row["primary_sample_id_as_received_by_glh"]},
        {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": glh_ods}},
         "type": {"coding": [{"system": V2_0203, "code": "STN"}]}, "value": row["glh_sample_consignment_number"]},
    ],
    # SpecimenAccessionNumber - the GLH LIMS's own number for the primary specimen (SPM-3).
    "accessionIdentifier": {"system": NGIS_PRIMARY_SAMPLE_ID_SYSTEM, "assigner": {"identifier": {"system": ODS_SYSTEM, "value": glh_ods}},
                            "type": {"coding": [{"system": V2_0203, "code": "ACSN"}]},
                            "value": row["primary_sample_id_in_glh_lims"]},
    # SampleSource, with the original sampleState kept as text. SNOMED CT goes last:
    # /transformToV2 builds SPM-4 from the last coding.
    "type": {"coding": [{"system": GEL_SAMPLE_SOURCE_SYSTEM, "code": sample_source},
                         {"system": SNOMED_SYSTEM,
                          "code": primary_sample_type_code,
                          "display": primary_sample_type_display}],
             "text": row["primary_sample_state"]},
    "subject": {"reference": patient_fullurl},
    "collection": {
        "quantity": {"value": float(row["dispatched_sample_volume_(ul)"]), "unit": "uL",
                     "system": "http://unitsofmeasure.org", "code": "uL"},
    },
    "receivedTime": row["primary_sample_received_date"] + "T00:00:00+00:00",
    "processing": [{"description": "Storage medium",
                    "procedure": {"coding": [{"system": GEL_STORAGE_MEDIUM_SYSTEM, "code": storage_medium}]}}],
    "container": [{"extension": [{"url": "http://hl7.org/fhir/5.0/StructureDefinition/extension-Specimen.container.device",
                                  "valueReference": {"type": "Device", "display": "DNA tube (FluidX 2D barcode)"}}],
                   "identifier": [
        {"system": NGIS_LAB_SAMPLE_ID_SYSTEM, "type": {"coding": [{"system": NWGMSA_CODESYSTEM, "code": "ZCID"}]}, "value": row["dispatched_sample_lsid"]}
    ]}],
}
# FillerSpecimenNumber - the dispatched DNA's number in the GLH LIMS (optional nationally).
if row["dispatched_sample_id_in_glh_lims"]:
    specimen["identifier"].append({"assigner": {"identifier": {"system": ODS_SYSTEM, "value": glh_ods}},
                                   "type": {"coding": [{"system": V2_0203, "code": "FILL"}]},
                                   "value": row["dispatched_sample_id_in_glh_lims"]})
# ordering_entity_id and received_sample_collection_date are optional, and a real GLH manifest
# export leaves both blank - leave the assigner/collectedDateTime out rather than guess them.
if row["ordering_entity_id"]:
    specimen["identifier"][0]["assigner"] = {"identifier": {"system": ODS_SYSTEM, "value": row["ordering_entity_id"]}}
if row["received_sample_collection_date"]:
    specimen["collection"]["collectedDateTime"] = row["received_sample_collection_date"] + "T00:00:00+00:00"
if sample_method:
    specimen["collection"]["method"] = {"coding": [{"system": GEL_METHOD_SYSTEM, "code": sample_method}]}
# dna_extraction_protocol is blank for germline blood (covaris/qiagen only apply to FFPE).
if row["dna_extraction_protocol"]:
    specimen["processing"].append({"description": "DNA extraction",
                                   "procedure": {"text": row["dna_extraction_protocol"]}})
if row["received_sample_topography"]:
    specimen["bodySite"] = {"text": row["received_sample_topography"]}

print(json.dumps(specimen, indent=2))

{
  "resourceType": "Specimen",
  "identifier": [
    {
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "PLAC"
          }
        ]
      },
      "value": "S26-1A01",
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "RR8"
        }
      }
    },
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "699N0"
        }
      },
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "STN"
          }
        ]
      },
      "value": "NEY GLH 2026-08-26"
    },
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "699N0"
        }
      },
      "type": {
        "coding

In [10]:
validate_resource(specimen, NWGMSA_SPECIMEN_PROFILE)

,severity,code,details,expression
5,error,structure,Specimen.container.extension: minimum required...,[Specimen.container[0]]
6,error,structure,Slice 'Specimen.container.extension:device' fo...,[Specimen.container[0]]
0,warning,code-invalid,None of the codings provided are in the value ...,[Specimen.identifier[1].type]
3,warning,not-found,A definition for CodeSystem 'https://gelreport...,[Specimen.collection.method.coding[0].system]
4,warning,not-found,A definition for CodeSystem 'https://gelreport...,[Specimen.processing[0].procedure.coding[0].sy...
7,warning,code-invalid,None of the codings provided are in the value ...,[Specimen.container[0].identifier[0].type]


**IG 2.2.1 update:** re-validating against the current NW-GMSA package (2.2.1; this
section was written against 2.0.3) adds two new `structure` errors.
`Specimen.container.extension:device` is now required (min 1), and nothing here
populates it yet. This comes from the IG's `Specimen` profile changing, not from this
notebook, and it's still open. The `ZCID`/`STN` issues described below are unchanged.

That `Specimen.type` error is gone now that `primary_sample_state` supplies a real
SNOMED CT code - the `warning` on `identifier[2].type` is unchanged from before: `STN`
isn't itself bound to the value set NW-GMSA's `Specimen` profile expects for that slice,
though it's still the correct v2-0203 code for what it represents (see the Specimen
section above). Coding `container[0].identifier[0].type` as `ZCID` adds two new issues
instead of resolving cleanly: an `error` because `ZCID` isn't a real v2-0203 code (it's
NE&Y Genomics' own local extension, as flagged above) so the validator can't resolve it
against that CodeSystem at all, plus the same kind of value-set `warning` `STN` gets.
(Since fixed: `ZCID` is now coded against the NW-GMSA `NWGMSA` CodeSystem.)
None of this stops the HL7 v2 transform below from running, and - unlike before the
`primary_sample_state` fix - that field now survives into the v2 output as a properly
coded `SPM-4` rather than silently disappearing (see the discrepancy check below).
`dispatched_sample_state` remains a separate, still-open gap: nothing in this worked
example builds it into any resource at all yet, coded or not.

### PractitionerRole: who approved the sub-order

`approved_by` is the GLH lead who approved the case for WGS (SOP Appendix 3, column 37).
When it's present, `ServiceRequest.requester` references a
[`PractitionerRole`](https://nw-gmsa.github.io/en/StructureDefinition-PractitionerRole.html)
rather than just the requesting GLH organisation:

- **`organization`** - the requesting GLH: `glh_laboratory_id`'s ODS code, via `GEL_GLH_ODS` (the
  profile requires `organization.identifier`), plus the lab's name.
- **`practitioner`** - `display` only, set to the `approved_by` text. The manifest gives
  a name but no identifier (no GMC/HCPC number), and the IG's `Practitioner` profile
  requires a `PractitionerIdentifier`, so a separate `Practitioner` resource can't be
  built. The IG's own `PractitionerRole` examples use the same display-only shape
  (e.g. "Result INTERPRETER" at North West Genomics `699X0`).

When `approved_by` is blank, `requester` stays a logical reference to the GLH
`Organization` (ODS code). The national manifest leaves `approved_by` optional, and NW
GMS doesn't require it. Every row in `Input/dWGS.csv` today has
`"Duty Scientist on behalf of NEY GMS"`, which describes a role rather than naming a
person, but that is what the manifest carries. The transformation spec doesn't map
`approved_by` to HL7 v2, so whether `/transformToV2` carries it (e.g. into `ORC-12`
Ordering Provider) still needs the transform re-run.


In [11]:
glh_organisation = {
    "identifier": {"system": ODS_SYSTEM, "value": glh_ods},
    "display": NEY_GLH_NAME[glh_ods],
}

practitioner_role = None
if row["approved_by"].strip():
    practitioner_role_fullurl = f"urn:uuid:{uuid4()}"
    practitioner_role = {
        "resourceType": "PractitionerRole",
        "meta": {"profile": [NWGMSA_PRACTITIONER_ROLE_PROFILE]},
        "practitioner": {"display": row["approved_by"]},  # name only - no identifier in the manifest
        "organization": glh_organisation,
    }
    requester = {"reference": practitioner_role_fullurl, "type": "PractitionerRole"}
    print(json.dumps(practitioner_role, indent=2))
else:
    requester = {**glh_organisation, "type": "Organization"}  # no approved_by - the GLH itself
    print("approved_by blank - requester is the GLH Organization:", json.dumps(requester))


{
  "resourceType": "PractitionerRole",
  "meta": {
    "profile": [
      "https://fhir.nwgenomics.nhs.uk/StructureDefinition/PractitionerRole"
    ]
  },
  "practitioner": {
    "display": "Duty Scientist on behalf of NEYGLH Medical"
  },
  "organization": {
    "identifier": {
      "system": "https://fhir.nhs.uk/Id/ods-organization-code",
      "value": "699N0"
    },
    "display": "Northern Genetics Service, Newcastle (NE&Y Genomics)"
  }
}


In [12]:
validate_resource(practitioner_role, NWGMSA_PRACTITIONER_ROLE_PROFILE) if practitioner_role else None

,severity,code,details,expression


### ServiceRequest

For this `LAB-35` sub-order, `requester` is `glh_laboratory_id` (NE&Y Genomics, ODS
`699N0` via `GEL_GLH_ODS`), via the `approved_by`
`PractitionerRole` above when there is one - the party actually placing *this*
order with NW Genomics - not `ordering_entity_id` (the *original* referring Trust).
`basedOn` references the parent `original-order` `ServiceRequest` above, and
`requisition` (Order Group Number, `ORC-4`) carries the parent's Order Filler Number,
`referral_id` (NGIS Referral Id, assigned by Genomics England `8J834`). Earlier passes
treated it as NE&Y Genomics' own group number assigned by `glh_laboratory_id`. The IG now
identifies it as the NGIS `referral_id` from TOMS. `supportingInfo` references the
`participant_type` `Observation` (the `OrderEntryQuestions` slice), when there is one. Every family member's
separate order shares the same `requisition`, so they can be matched by that value too.
The referring Trust is the parent's `requester`, and is also kept as the `Specimen`
identifier assigner above - the original order placer's involvement doesn't disappear,
it just isn't `ServiceRequest.requester` for *this* message.

`intent` is `filler-order`, not the plain `order` `03-laboratory-order-from-csv.ipynb`
used - this `ServiceRequest` is NW Genomics (the filler) restating NE&Y Genomics'
sub-order, not an original order in its own right.

`intent` isn't always `filler-order`, though: when `retrospective_sample` is
`"Retrospective"` rather than `"New"`, NW Genomics is reflexing further testing off a
specimen it already has - not NE&Y Genomics placing a fresh sub-order - so `intent`
switches to `reflex-order`. Every row in `Input/dWGS.csv` today is `"New"`, so this
worked example's output doesn't change, but the field-mapping table above now records
the distinction.

Two identifiers, each following its own NW-GMSA identifier profile:

- **`requisition`** (Order Group Number, `ORC-4`) -
  [NGIS Referral Id](https://nw-gmsa.github.io/en/StructureDefinition-NGISReferralId.html):
  `referral_id`, system `https://genomicsengland.co.uk/healthcare/referral-id`,
  `type` `PGN` (from its base `OrderGroupNumber` profile), assigned by Genomics England
  `8J834`. It's shared by every participant in a family, so on its own it does **not**
  identify one order.
- **`identifier`** (Order Placer Number) - the `OrderIdentifier` slice:
  `dispatched_sample_lsid` (e.g. `4000000001`), `type` `PLAC`, assigned by the sending
  lab's ODS code (`glh_ods`, `699N0` here). This is the interim Placer Order Number the
  transformation spec v0.5 adopts, since NHS England won't add a `placer_order_number`
  field to the manifest. It replaces the
  earlier NGIS Composite Order Number (`f"{referral_id}-{patient_ngis_id}"`, `FILL`,
  `8J834`), which couldn't be the placer order number because it isn't unique. A real GLH
  manifest export lists a cancer referral's germline and tumour samples as two rows for the
  same patient, so both rows have the same `referral_id` and `patient_ngis_id`. The LSID is
  the barcode of one dispatched tube, so it's unique per manifest row. Each row, including
  each half of a tumour-normal pair, is therefore its own order with its own `Specimen`. The
  cell below asserts that across every manifest row, not just this one. The same LSID also
  stays on `Specimen.container.identifier` (`ZCID`). The profile defines no `system`, so none
  is set.

In [13]:
service_request_fullurl = f"urn:uuid:{uuid4()}"
# retrospective_sample == "Retrospective" means this sub-order is NW Genomics reflexing
# testing off an already-received specimen, not NE&Y Genomics placing a fresh filler
# order - every row in Input/dWGS.csv today is "New", so this stays "filler-order" in
# practice, but the switch is real.
service_request_intent = "reflex-order" if row["retrospective_sample"] == "Retrospective" else "filler-order"


# Order Placer Number = dispatched_sample_lsid (interim Placer Order Number, spec v0.5). It must be unique per
# manifest row - referral_id/patient_ngis_id aren't (Duo/Trio families, and a cancer referral's
# germline + tumour rows for the same patient, share them).
assert dwgs_df.dispatched_sample_lsid.is_unique, \
    dwgs_df[dwgs_df.dispatched_sample_lsid.duplicated(keep=False)].dispatched_sample_lsid.tolist()

order_identifier = {  # ORDER_IDENTIFIER_PROFILE (ServiceRequest.identifier:OrderIdentifier)
    "type": {"coding": [{"system": V2_0203, "code": "PLAC"}]},
    "value": row["dispatched_sample_lsid"],
    "assigner": {"identifier": {"system": ODS_SYSTEM, "value": glh_ods}},  # the sending lab
}

service_request = {
    "resourceType": "ServiceRequest",
    "status": "active",
    "intent": service_request_intent,
    "category": [{"coding": [{"system": "http://snomed.info/sct", "code": "116148004"}]}],
    "code": {"coding": [{"system": GENOMIC_TEST_DIRECTORY_SYSTEM, "code": row["clinical_indication_test_type_id"]}]},
    "requisition": referral_identifier,  # NGIS_REFERRAL_ID_PROFILE - the parent's Order Filler Number
    "identifier": [order_identifier],
    "basedOn": [{"reference": parent_service_request_fullurl, "type": "ServiceRequest"}],  # original-order
    "subject": {"reference": patient_fullurl},
    "requester": requester,  # PractitionerRole (approved_by) or the GLH Organization - see above
    "specimen": [{"reference": specimen_fullurl, "type": "Specimen"}],
}
if participant_type_fullurl:
    service_request["supportingInfo"] = [{"reference": participant_type_fullurl, "type": "Observation"}]  # OrderEntryQuestions slice

# clinical_information is optional - leave note out rather than emit an empty string.
if row["clinical_information"]:
    service_request["note"] = [{"text": row["clinical_information"]}]  # SOP v1.0 item 42 -> NTE-3

print(json.dumps(service_request, indent=2))

{
  "resourceType": "ServiceRequest",
  "status": "active",
  "intent": "filler-order",
  "category": [
    {
      "coding": [
        {
          "system": "http://snomed.info/sct",
          "code": "116148004"
        }
      ]
    }
  ],
  "code": {
    "coding": [
      {
        "system": "https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirectory",
        "code": "R14.1"
      }
    ]
  },
  "requisition": {
    "system": "https://genomicsengland.co.uk/healthcare/referral-id",
    "value": "r26000000201",
    "assigner": {
      "identifier": {
        "system": "https://fhir.nhs.uk/Id/ods-organization-code",
        "value": "8J834"
      }
    },
    "type": {
      "coding": [
        {
          "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
          "code": "PGN"
        }
      ]
    }
  },
  "identifier": [
    {
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "PLAC"

In [14]:
validate_resource(service_request, NWGMSA_SERVICE_REQUEST_PROFILE)

,severity,code,details,expression
0,information,informational,This element does not match any known slice de...,[ServiceRequest.supportingInfo[0]]


**IG 2.2.1 update:** this now validates cleanly. The `authoredOn` cardinality error
described below no longer appears against the current package, so `ServiceRequest.authoredOn`
appears to have been relaxed in the profile. The paragraph is kept for history.
Adding `basedOn` (the parent `original-order`) and pointing `supportingInfo` at the
`participant_type` `Observation` instead of a `FamilyGeneticGroup` introduced no new
errors. The `information` message on `supportingInfo[0]` is because a single resource
validated on its own can't resolve the `Observation` it references; the `Bundle`
validation below resolves it and doesn't report it.

The remaining `error` (`ServiceRequest.authoredOn: minimum required = 1`) is the
same pre-existing NW-GMSA profile gap `04-laboratory-report-fhir-from-hl7v2.ipynb`
already found and confirmed against this repo's real production output: nothing in
`dWGS.csv` records when the order was authored, so - as there - it's left out rather
than filled with an invented date.

### MessageHeader and the Bundle

`sender` is NE&Y Genomics (`699N0`, looked up from `glh_laboratory_id` `yne`, this `LAB-35` sub-order's Order Placer);
`destination` is NW Genomics (`699X0`, the Order Filler). `Bundle.identifier`/
`.timestamp` are mandatory on the
[`Bundle` (message) profile](https://nw-gmsa.github.io/en/StructureDefinition-BundleMessage.html),
as throughout this series. The parent `ServiceRequest` and the `Observation`s go in as
their own entries, so they are reachable forwards from `MessageHeader.focus` (the
sub-order) via `basedOn` and `supportingInfo`, like every other entry. The
`PractitionerRole`, when there is one, is reachable the same way, via
`ServiceRequest.requester`.

In [15]:
message_header = {
    "resourceType": "MessageHeader",
    "eventCoding": {"system": "http://terminology.hl7.org/CodeSystem/v2-0003", "code": "O21"},
    "sender": {"identifier": {"system": ODS_SYSTEM, "value": glh_ods}},
    "destination": [{"endpoint": "https://fhir.nwgenomics.nhs.uk/Endpoint/RIE",
                      "receiver": {"identifier": {"system": ODS_SYSTEM, "value": NW_GLH_ODS}}}],
    "source": {"endpoint": "https://fhir.nwgenomics.nhs.uk/Endpoint/NEYGenomics", "software": "NE&Y Genomics"},
    "focus": [{"reference": service_request_fullurl}],
}

order_bundle = {
    "resourceType": "Bundle",
    "identifier": {"value": f"urn:uuid:{uuid4()}"},
    "timestamp": datetime.now().astimezone().strftime("%Y-%m-%dT%H:%M:%S+00:00"),
    "type": "message",
    "entry": [
        {"fullUrl": f"urn:uuid:{uuid4()}", "resource": message_header},
        {"fullUrl": patient_fullurl, "resource": patient},
        {"fullUrl": specimen_fullurl, "resource": specimen},
        {"fullUrl": parent_service_request_fullurl, "resource": parent_service_request},
        *[{"fullUrl": full_url, "resource": observation} for full_url, observation in observations.items()],
        {"fullUrl": service_request_fullurl, "resource": service_request},
    ],
}

if practitioner_role:
    order_bundle["entry"].insert(-1, {"fullUrl": practitioner_role_fullurl, "resource": practitioner_role})

order_filename = f"dWGS_{row['referral_id']}.json"
with open("Input/FHIR/O21/" + order_filename, "w") as f:
    json.dump(order_bundle, f, indent=2)

print("Saved Input/FHIR/O21/" + order_filename)

Saved Input/FHIR/O21/dWGS_r26000000201.json


In [16]:
outcome_path = "Results/FHIR/O21/" + order_filename + "-OperationOutcome.json"
subprocess.run(
    [
        "java", "-jar", "validator_cli.jar", "Input/FHIR/O21/" + order_filename,
        "-version", "4.0.1", "-ig", "package.tgz",
        "-bundle", "ServiceRequest:0", NWGMSA_SERVICE_REQUEST_PROFILE, "-tx", "n/a",
        "-output", outcome_path, "-output-style", "json",
    ],
    capture_output=True,
)
with open(outcome_path) as f:
    issues_df(json.load(f))

## HL7 v2 worked example

Same `transformToV2` tooling endpoint used throughout this series. An earlier draft of
this `Bundle` - before the `Patient.identifier[1].type` fix above (`coding` + `text`
together, which turned out to violate a fixed value the profile pins that slice to) -
got a `500` back with an empty body from this same call. That's worth knowing even
though it's fixed now: a structural profile violation on `Patient` was enough to fail
the *whole* transform, not just that one field, with nothing in the response to say
why - the `FHIR Validation.ipynb`-style checks above are what actually located it.

### Expected `OML^O21` order numbers and assigners (hand-built)

Before calling the transform, this cell builds the `OML^O21` the FHIR above *should*
become, straight from the notebook's own resources, so the order numbers and their
assigners can be checked. It follows the IG's dWGS field mapping (`dWGS.md`) and the
transformation spec. **This is an expected message, not transform output.** The
transform's actual output comes from the call below.

| v2 field | Value | From |
|---|---|---|
| `PID-3` (NGIS) | `p…^^^8J834^PI` | `Patient.identifier` (PI, assigner Genomics England `8J834`) |
| `ORC-2` / `OBR-2` Placer Order Number | `<LSID>^<GLH ODS>` | `ServiceRequest.identifier:OrderIdentifier` - `dispatched_sample_lsid` (`PLAC`, assigner = sending lab ODS) |
| `ORC-3` / `OBR-3` Filler Order Number | *empty* | assigned by NW Genomics (the Order Filler) once it accepts the order |
| `ORC-4` Placer Group Number | `r…^8J834` | `ServiceRequest.requisition` - NGIS Referral Id (`PGN`, assigner `8J834`) - IG only |
| `ORC-8` Parent | `^r…&8J834` | `basedOn` - the parent `original-order`'s Order Filler Number (`FILL`, assigner `8J834`) - IG's provisional mapping |
| `ORC-21` Ordering Facility | `^^<GLH ODS>^^^ODS` | requesting GLH (`PractitionerRole.organization`) |
| `OBX` `FAMILY_STRUCTURE` / `PARTICIPANT_TYPE` | values | the `FamilyStructure` `Observation` (on the parent, sent with the sub-order since only the filler order is sent) and the `ParticipantType` `Observation` |
| `NTE-3` | `clinical_information` | `ServiceRequest.note` (only when present) |
| `SPM-2` Specimen ID | `<LSID>` | `Specimen.container.identifier` - `dispatched_sample_lsid` only |
| `SPM-3` Specimen Parent IDs | `^<LIMS id>&<GLH ODS>` | `Specimen.accessionIdentifier` - `primary_sample_id_in_glh_lims` (`ACSN`) |

Three points to note, all against the transformation spec v0.5:

- **`ORC-2`/`OBR-2` carry the LSID.** The spec uses `dispatched_sample_lsid` as the
  interim Placer Order Number and requires the same value in `ORC-2.1`, `OBR-2.1`,
  `SPM-2.1` and the `DISPATCHED_SAMPLE_LSID` OBX. The NW canonical `OML^O21` carries it in
  `ORC-2.1`, `OBR-2.1` and `SPM-2.1` only: the RIE adds the `DISPATCHED_SAMPLE_LSID` (and
  `NGIS_REFERRAL_ID`) OBX when it converts the message to iGene's `ORM^O01`, so neither is
  in this expected message. Here the namespace is the sending lab's
  ODS code. The spec's worked example uses `RR8` instead, and lists the LSID's assigning
  authority as an open decision. `ORC-3`/`OBR-3` (Filler Order Number) are left for NW
  Genomics to assign; the earlier NGIS Composite Order Number (`r…-p…^8J834`) that sat
  there isn't unique per order, so it's no longer built.
- **The NGIS patient identifier is `PI`.** The spec writes it `^^^NGIS^MR`. This notebook
  codes it `PI` (it isn't a hospital MRN) and keeps the IG's convention of the assigner's
  ODS code as the assigning authority, so `^^^8J834^PI`.
- **`ORC-4`/`ORC-8` are the IG's, not the spec's.** The IG maps `ServiceRequest.requisition`
  (the NGIS Referral Id) to `ORC-4`, and the parent order's Filler Order Number to `ORC-8`
  (provisionally). The spec v0.5 sends `referral_id` only as the `NGIS_REFERRAL_ID` OBX,
  which the RIE adds for iGene.


In [17]:
def ei(identifier):
    """HL7 v2 EI (Entity Identifier^Namespace ID) from a FHIR Identifier - namespace = assigner ODS code."""
    return f"{identifier['value']}^{identifier['assigner']['identifier']['value']}"


def v2_date(value):
    return value.replace("-", "")


def v2_datetime(value):
    return value[:10].replace("-", "") + "000000+0000"


ngis_identifier = next(i for i in patient["identifier"] if i.get("assigner"))
placer_order_number = ei(service_request["identifier"][0])   # LSID, assigned by the sending lab
placer_group_number = ei(service_request["requisition"])     # NGIS Referral Id
parent_filler_number = ei(referral_filler_number).replace("^", "&")  # parent original-order (ORC-8 EIP filler part)
accession = specimen["accessionIdentifier"]
spm_accession = f"{accession['value']}&{accession['assigner']['identifier']['value']}"  # SPM-3 (EIP filler part)
lsid = specimen["container"][0]["identifier"][0]["value"]  # SPM-2.1 - the only SPM-2 value
specimen_type = next(c for c in specimen["type"]["coding"] if c["system"] == SNOMED_SYSTEM)

expected_segments = [
    f"MSH|^~\\&|NEYGenomics|{glh_ods}|RIE|{NW_GLH_ODS}|{datetime.now().strftime('%Y%m%d%H%M%S')}+0000||OML^O21^OML_O21|{order_bundle['identifier']['value']}|T|2.5.1|||AL",
    f"PID|1||{row['patient_nhs_number']}^^^NHS^NH~{ngis_identifier['value']}^^^{ngis_identifier['assigner']['identifier']['value']}^PI"
    f"||{patient_surname}^{patient_forename}^^^^^L||{v2_date(row['patient_date_of_birth'])}",
    f"ORC|NW|{placer_order_number}||{placer_group_number}||||^{parent_filler_number}" + "|" * 13 + f"^^{glh_ods}^^^ODS",
    f"OBR|1|{placer_order_number}||{row['clinical_indication_test_type_id']}^^England-GenomicTestDirectory",
]
if "note" in service_request:
    expected_segments.append(f"NTE|1||{service_request['note'][0]['text']}")
# NGIS_REFERRAL_ID and DISPATCHED_SAMPLE_LSID are added by the RIE for iGene's ORM^O01, not in the OML^O21.
for n, (code_, value) in enumerate([(code_, value) for code_, value in [
    ("FAMILY_STRUCTURE", row["family_structure"]),
    ("PARTICIPANT_TYPE", row["participant_type"]),
] if value], start=1):
    expected_segments.append(f"OBX|{n}|ST|{code_}||{value}||||||F")
expected_segments.append(
    f"SPM|1|{lsid}|^{spm_accession}|{specimen_type['code']}^{specimen_type.get('display', '')}^SCT"
    + "|" * 13 + f"{v2_datetime(row['received_sample_collection_date']) if row['received_sample_collection_date'] else ''}|{v2_datetime(row['primary_sample_received_date'])}"
)
expected_v2 = "\r".join(expected_segments) + "\r"

expected_filename = order_filename.replace(".json", "-expected.txt")
with open("Output/V2/O21/" + expected_filename, "w", newline="") as f:
    f.write(expected_v2)

print("Saved Output/V2/O21/" + expected_filename + " (hand-built expectation, not transform output)\n")
print(expected_v2.replace("\r", "\n"))


Saved Output/V2/O21/dWGS_r26000000201-expected.txt (hand-built expectation, not transform output)

MSH|^~\&|NEYGenomics|699N0|RIE|699X0|20261109135825+0000||OML^O21^OML_O21|urn:uuid:18294837-c6c0-45d6-b9e2-8eb4b5fd2b22|T|2.5.1|||AL
PID|1||9737383222^^^NHS^NH~p26000000101^^^8J834^PI||LEEDS^Rob^^^^^L||19780117
ORC|NW|4000000001^699N0||r26000000201^8J834||||^r26000000201&8J834|||||||||||||^^699N0^^^ODS
OBR|1|4000000001^699N0||R14.1^^England-GenomicTestDirectory
NTE|1||Suspected inherited neurodevelopmental condition.
OBX|1|ST|FAMILY_STRUCTURE||Singleton||||||F
OBX|2|ST|PARTICIPANT_TYPE||Proband||||||F
SPM|1|4000000001|^S26-1A01&699N0|258580003^Whole blood specimen (specimen)^SCT|||||||||||||20260818000000+0000|20260820000000+0000



In [18]:
with open("Input/FHIR/O21/" + order_filename, "rb") as f:
    order_json = f.read()

rV2 = requests.post(toolsServer + "/transformToV2", data=order_json, verify=False, headers={"Content-Type": "application/fhir+json"})

print(rV2.status_code)

v2_filename = order_filename.replace(".json", ".txt")
with open("Output/V2/O21/" + v2_filename, "w") as f:
    f.write(rV2.text)

print(rV2.text)

200




In [19]:
def v2_field(message, segment, field):
    """First occurrence of SEGMENT-field (1-based, MSH not special-cased - only used on ORC/OBR/PID)."""
    for seg in message.replace("\n", "\r").split("\r"):
        parts = seg.split("|")
        if parts[0] == segment:
            return parts[field] if field < len(parts) else ""
    return None


order_number_fields = [("PID", 3), ("ORC", 2), ("ORC", 3), ("ORC", 4), ("ORC", 8), ("OBR", 2), ("OBR", 3)]
pd.DataFrame([
    {"field": f"{seg}-{n}", "expected": v2_field(expected_v2, seg, n), "transform": v2_field(rV2.text, seg, n),
     "match": v2_field(expected_v2, seg, n) == v2_field(rV2.text, seg, n)}
    for seg, n in order_number_fields
])


,field,expected,transform,match
0,PID-3,9737383222^^^NHS^NH~p26000000101^^^8J834^PI,9737383222^^^NHS^NH~p26000000101^^^8J834^PI,True
1,ORC-2,4000000001^699N0,4000000001^699N0,True
2,ORC-3,,,True
3,ORC-4,r26000000201^8J834,r26000000201^8J834,True
4,OBR-2,4000000001^699N0,4000000001^699N0,True
5,OBR-3,,,True


### Sending the order onward

`Testing.ipynb`'s `O21`/`O01` loops already `POST` their bundles to `FHIR_SERVER`'s
`$process-message` endpoint after building them, using an OAuth2 client-credentials
token (`OAUTH2_TOKEN`/`CLIENT_ID`/`CLIENT_SECRET` from `.env`) - this worked example
does the same for `order_bundle` below, something an earlier version of this section
said wasn't exercised here.

That earlier version's reasoning doesn't hold up: a first attempt at this exact call
hung indefinitely with no response at all, long enough to look like a genuine
cross-organisation firewall block between NE&Y Genomics and NW Genomics'
`FHIR_SERVER`. But `V2_TOOLS`'s own `/transformToV2` endpoint was independently
timing out the same way at the same time, then answering normally again a few
checks later with no change on this notebook's side - so both endpoints were
cycling between healthy and unresponsive, and the `$process-message` hang was that
same infrastructure flakiness, not a network boundary. Repeating the call once
`V2_TOOLS` had settled succeeded immediately (see below). **Still worth tracking for
the real implementation**: production delivery for this inter-organisation route is
expected to go via a message queue (AWS SQS) rather than a direct synchronous HTTP
POST regardless, so intermittent reachability like this wouldn't be this route's
problem in practice.

In [14]:
from requests.auth import HTTPBasicAuth

fhirServer = os.getenv("FHIR_SERVER")
tokenUrl = os.getenv("OAUTH2_TOKEN")
clientId = os.getenv("CLIENT_ID")
clientSecret = os.getenv("CLIENT_SECRET")

token_response = requests.post(
    tokenUrl,
    auth=HTTPBasicAuth(clientId, clientSecret),
    verify=False,
    data={"grant_type": "client_credentials", "scope": "system/*.*"},
    headers={"Content-Type": "application/x-www-form-urlencoded"},
)
token = token_response.json()["access_token"]

rProcessMessage = requests.post(
    fhirServer + "$process-message",
    data=order_json,
    verify=False,
    headers={"Content-Type": "application/fhir+json", "Authorization": "Bearer " + token},
)

print(rProcessMessage.status_code)
print(json.dumps(rProcessMessage.json(), indent=2))


/Users/kevinmayfield/github/MFT/Testing/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/kevinmayfield/github/MFT/Testing/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


200
{
  "resourceType": "Bundle",
  "entry": [
    {
      "fullUrl": "urn:uuid:15d00b47-ab96-4a35-a584-64d8d0e9dbec",
      "resource": {
        "resourceType": "MessageHeader",
        "destination": [
          {
            "receiver": {
              "identifier": {
                "system": "https://fhir.nhs.uk/Id/ods-organization-code",
                "value": "699C0"
              }
            }
          }
        ],
        "eventCoding": {
          "code": "O21",
          "system": "http://terminology.hl7.org/CodeSystem/v2-0003"
        },
        "response": {
          "code": "ok",
          "details": {
            "reference": "urn:uuid:34872447-65fa-48f2-874b-91eac794eeda"
          },
          "identifier": "urn:uuid:a22edb38-f8c1-4af3-99ad-9501f93a72e4"
        },
        "sender": {
          "identifier": {
            "system": "https://fhir.nhs.uk/Id/ods-organization-code",
            "value": "X24"
          }
        },
        "source": {
          "end

`$process-message` accepts `order_bundle` and returns `200` with `response.code`
`"ok"` - the RIE confirms receipt rather than rejecting or silently dropping it. Its
`OperationOutcome` carries an `information`-severity `diagnostics` entry, not an
error - the RIE's own re-derived v2 message.

That echoed message matches `/transformToV2`'s own output above field-for-field
(same `SPM-2` components, same `OBX|3|ST|ZCID` container identifier, same trailing
consignment-number field) - so the RIE's `$process-message` route and `V2_TOOLS`'s
standalone `/transformToV2` endpoint are doing the same mapping for this input, not
two different ones. (An earlier run within this same session, before `V2_TOOLS`
briefly went unresponsive and came back - see above - showed the two disagreeing:
`/transformToV2` alone dropped `primary_sample_id_in_glh_lims`, the `FILL`-coded
LIMS Sample Identifier, that `$process-message`'s RIE mapping carried through fine.
Whatever changed server-side during that gap fixed `/transformToV2` to match - see
the discrepancy check below.)

One difference does remain: the response's `MessageHeader.sender` is a fixed RIE
identity (ODS `X24`), not an echo of `order_bundle`'s own `sender` - it's replying as
itself, not relaying the original message header back.

## Checking the transform

A `200` back doesn't mean everything survived the trip - check every field value this
notebook put into the FHIR `Bundle` for whether it actually shows up somewhere in the
v2 message `/transformToV2` returned above.

In [21]:
check_values = {
    "referral_id (requisition)": row["referral_id"],
    "Order Placer Number - LSID (identifier)": order_identifier["value"],
    "clinical_indication_test_type_id (ServiceRequest.code)": row["clinical_indication_test_type_id"],
    "patient_nhs_number": row["patient_nhs_number"],
    "patient_forename (PID-5.2)": patient_forename,
    "patient_surname (PID-5.1)": patient_surname,
    "ordering_entity_id (requester ODS)": row["ordering_entity_id"],
    "primary_sample_id_as_received_by_glh": row["primary_sample_id_as_received_by_glh"],
    "primary_sample_id_in_glh_lims (accessionIdentifier)": row["primary_sample_id_in_glh_lims"],
    "dispatched_sample_id_in_glh_lims (identifier FILL)": row["dispatched_sample_id_in_glh_lims"],
    "primary_sample_state (Specimen.type coding)": primary_sample_type_display,
    "primary_sample_state Method (collection.method)": sample_method or "",
    "primary_sample_state StorageMedium (processing)": storage_medium,
    "dispatched_sample_lsid (container.identifier)": row["dispatched_sample_lsid"],
    "dispatched_sample_state (Specimen.type text)": row["dispatched_sample_state"],
    "dna_extraction_protocol (processing)": row["dna_extraction_protocol"],
    "dispatched_sample_volume_(ul) (collection.quantity)": row["dispatched_sample_volume_(ul)"],
    "glh_sample_consignment_number": row["glh_sample_consignment_number"],
    "family_structure (parent supportingInfo Observation)": row["family_structure"],
    "participant_type (supportingInfo Observation)": row["participant_type"],
    "clinical_information (ServiceRequest.note)": row["clinical_information"],
    "approved_by (PractitionerRole.practitioner.display)": row["approved_by"],
    "NE&Y Genomics sender ODS": glh_ods,
    "NW Genomics destination ODS": NW_GLH_ODS,
}

pd.DataFrame(
    [{"field": field, "value": value, "present_in_v2": value in rV2.text} for field, value in check_values.items()]
)

,field,value,present_in_v2
0,referral_id (requisition),r26000000201,True
1,Order Placer Number - LSID (identifier),4000000001,True
2,clinical_indication_test_type_id (ServiceReque...,R14.1,True
3,patient_nhs_number,9737383222,True
4,patient_forename (PDS),Rob,True
5,patient_surname (PDS),LEEDS,True
6,ordering_entity_id (requester ODS),RR8,True
7,primary_sample_id_as_received_by_glh,S26-1A01,True
8,primary_sample_id_in_glh_lims,S26-1A01,True
9,primary_sample_state (Specimen.type coding),Whole blood specimen (specimen),True


> **Re-run (Family Structure / PI / SPM pass).** `family_structure` is now an
> `ObservationOrder` about the `Patient`, referenced from the parent `original-order`'s
> `supportingInfo` (a `Group` subject fails the IG 2.2.6 `ObservationOrder` profile). `patient_ngis_id` is `PI` rather than `MR`. The hand-built expected v2
> follows the RIE's NW canonical `OML^O21`: `PID-3` `^^^8J834^PI`, the LSID as the only
> `SPM-2` value, the accession number in `SPM-3`, and only the `FAMILY_STRUCTURE` and
> `PARTICIPANT_TYPE` OBX (the RIE adds `NGIS_REFERRAL_ID` and `DISPATCHED_SAMPLE_LSID` for
> iGene's `ORM^O01`). The FHIR build, validation and expected v2 were re-run against IG
> package 2.2.6. `V2_TOOLS` and `FHIR_SERVER` weren't re-run, so the transform output,
> `$process-message` and the two check cells are from an earlier pass.

> **Re-run (parent ServiceRequest pass).** The `FamilyGeneticGroup` is replaced by a parent
> `original-order` `ServiceRequest` (`referral_id` as its `FILL` Order Filler Number,
> referenced from `basedOn`, `subject` a logical `Group` reference by `referral_id`) and an
> `ObservationOrder` for `participant_type` (sub-order `supportingInfo`); `family_structure`
> is no longer in the FHIR order. `primary_sample_id_in_glh_lims` moved to
> `Specimen.accessionIdentifier` (`ACSN`, `SPM-2.2`) and `dispatched_sample_id_in_glh_lims`
> is now the `FILL` identifier. The FHIR build, validation and hand-built expected v2 (now
> with `ORC-8`) were re-run, against IG package 2.2.6: all six dWGS `Bundle`s validate
> with no errors.
> `V2_TOOLS` was still unreachable, so the transform output and the two check cells above
> are from the `FamilyGeneticGroup` `Bundle`.

> **Re-run (SOP v1.0 / transformation spec v0.5 pass).** `Input/dWGS.csv` now carries
> all 42 manifest columns, so the name and family/clinical fields come from the manifest,
> and the NGIS Patient ID is `MR` rather than `PI`. The FHIR build, validation and
> hand-built expected v2 above were re-run; the only changes to the `O21` fixtures are
> `PI` -> `MR` and the EDI test patients' surname case. `V2_TOOLS` was unreachable for
> this pass, so the `/transformToV2` output, order-number comparison and discrepancy
> check cells above still show the previous pass's results (with `PI`).

> **Re-run (LSID Order Placer Number pass, IG 2.2.1).** `/transformToV2` and the two
> check cells above were re-run against the `FamilyGeneticGroup`/LSID `Bundle`.
> `$process-message` (`FHIR_SERVER`) was not re-run, so its output above is still from the
> earlier `Observation`-based `Bundle`. The order-number comparison now matches on
> `ORC-2`/`OBR-2` (`<LSID>^<GLH ODS>`, e.g. `4000000001^699N0`), `ORC-3`/`OBR-3` (empty)
> and `ORC-4` (`r…^8J834`). The one mismatch is `PID-3`, where the transform emits an
> empty repetition (`…^NH~~p…`) that the hand-built expectation doesn't. The transform
> did this before this pass too. The discrepancy check above shows the `Group` isn't
> carried into v2: `family_structure` and `participant_type` both come back `False` (no
> `OBX` `FAMILY_STRUCTURE`/`PARTICIPANT_TYPE`), and so does `approved_by`.
> `dispatched_sample_state` and `dispatched_sample_volume_(ul)` are also `False`. The
> paragraph below describes the earlier `Observation` runs.

Coding the `Specimen` identifiers' `type` (`PLAC`/`FILL`/`STN`) and `Specimen.type`
itself (a real SNOMED CT code from `Input/dWGS.csv` now, not free text), plus coding
`container.identifier.type` as `ZCID`, now accounts for every field this table checks
except one. `PLAC` (Received Sample Identifier), `STN` (Shipment Tracking Number),
`FILL` (LIMS Sample Identifier), `Specimen.type`, and `container.identifier` (the
dispatched-sample barcode, `ZCID`-coded above) all come through -
`ordering_entity_id`, `primary_sample_id_as_received_by_glh`,
`primary_sample_id_in_glh_lims`, `primary_sample_state`, and
`dispatched_sample_lsid` all flip from `False` to `True` across this notebook's
passes. `FILL` was the last holdout: an earlier run within this same session still
recorded it `False` here even though `$process-message`'s RIE mapping already carried
it through (see above) - a real difference between the two entry points at the time,
not a coding gap on this notebook's side. Something changed server-side between that
run and this one (`V2_TOOLS` was intermittently unresponsive in between - see above)
that brought `/transformToV2` alone in line with `$process-message`. The two "ask at
order" `Observation.valueCodeableConcept`s (`family_structure`, `participant_type`)
flipped the same way, for the same reason: nothing in this worked example's own code
changed across those runs, so both flips reflect the live transform service's
evolving behaviour, not this notebook - a reminder that this check exercises a real,
evolving system rather than a fixed target. This pass adds two new fields, and finds a
genuine gap rather than a flip: `dna_extraction_protocol` (`Specimen.collection.method`)
and `dispatched_sample_volume_(ul)` (`Specimen.collection.quantity`) both build and
validate cleanly, but neither survives into the v2 output - `SPM-7`/`SPM-12` stay blank
rather than carrying `"Qiagen"`/`120`, unlike everything else `Specimen.collection`
already carried (`collectedDateTime` lands correctly). Whether that's the same kind of
transform-mapping gap `FILL` briefly was, or something this pass's fields never had
support for, isn't something this single run can tell. `dispatched_sample_state` (not
built into any resource at all, coded or not) remains the one field that vanishes for a
known, unrelated reason. Every genuinely coded value elsewhere (NHS number, ODS codes
on `MessageHeader`/`requester`, the Genomic Test Directory code) comes through fine
throughout.

## Look forward: what comes back

This order is one half of a closed loop - [NW-GMSA's ILW
page](https://nw-gmsa.github.io/en/ILW.html) pairs this `LAB-35` sub-order with a
`LAB-36` result coming back the other way. Which shape that result takes depends on
what the requester actually needs:

- **A PDF report** - `LAB-3` (`ORU^R01`/FHIR `R01` Message), the narrative report
  `04-laboratory-report-fhir-from-hl7v2.ipynb` builds.
- **Structured variants** - `LAB-5` (`ORU^R01`/FHIR `R01` Message), the discrete
  genomic findings `05-test-results-from-vcf.ipynb` builds.
- **Both together** - an XD-LAB/FHIR Document, combining a PDF and structured results
  in one document `Bundle`, which `06-eu-laboratory-report-fhir-document.ipynb` builds.
  This is the shape most likely to matter here: it's the format NHS England's Unified
  Genomic Record (UGR) is expected to use.

## Closing note

This is a preparation/analysis notebook, not a settled spec. Running the worked example
above already surfaced real findings this notebook's earlier analysis alone hadn't:
corrections to the field-mapping table (`family_structure`/`participant_type` moving
from a guessed extension to `Observation`/`supportingInfo`, and then - once the IG
introduced it - to a `FamilyGeneticGroup` linked by `requisition`, `glh_laboratory_id` moving
off `ServiceRequest.requester` onto `MessageHeader.sender`), a structural `Patient`
mistake that failed the *entire* v2 transform with no explanation until the FHIR
validation step caught it, and - via the discrepancy check - confirmation that
text-only values reliably vanish in that same transform while coded ones survive,
a live example of the target transform service itself changing mid-session
(`FILL`, `family_structure`, and `participant_type` all flipping from vanishing to
surviving between two runs of the exact same code), and - from this pass's two new
`Specimen.collection` fields - a fresh, still-open gap of the same shape: correctly
built and validated FHIR that doesn't survive into `SPM-7`/`SPM-12` regardless - not
something coding the identifiers was expected to leave open, but a real property of
testing against a live system rather than a fixed target. Exactly the kind of thing this notebook is *for*.
Expect further passes to
keep finding gaps like these, which may in turn mean `Input/dWGS.csv` itself needs
updating (new columns, corrected values) - which would then change this worked example
again. That loop is the intended way this notebook gets used, not a sign something went
wrong.

## Additional data: fields not in this worked example

`Input/dWGS.csv` covers all 42 SOP v1.0 manifest fields. NHS England won't add
`placer_order_number` or `patient_medical_record_number`, so the worked example uses the
LSID and the NGIS Patient ID in their place, as the transformation spec v0.5 does (see
[Sample manifest data](#Sample-manifest-data)). Beyond those, further fields may be added to the
payload - mainly to help the *original* order placer (the referring NHS Trust) and
order filler (the Requesting Genomic Laboratory) track this sub-order, rather than
anything NW Genomics (and Genomics England Limited) itself needs to fulfil it. None of these are built into the
worked example above - documented here as a reference for a future pass, not
implemented:

- **Medical Record Number (MRN)**, a.k.a. hospital number - usually a Trust's primary
  patient identifier. The manifest still doesn't carry the referring Trust's own MRN, and
  `patient_ngis_id` is sent as `PI`, not as an MRN-equivalent (the spec v0.5's `MR`). See
  [`Patient`](https://nw-gmsa.github.io/en/StructureDefinition-Patient.html).
- **Account Number / Visit Number**, a.k.a. Hospital Spell Identifier - links every
  diagnostic order from the same episode together, the same role `referral_id` plays
  for this sub-order. See the `encounter` element on
  [`ServiceRequest`](https://nw-gmsa.github.io/en/StructureDefinition-ServiceRequest.html).
  Can be very useful to the ordering NHS Trust, and in some cases is required by them.
- **Order Placer Number** - an identifier assigned by `ordering_entity_id` (the
  referring Trust). Helps the Trust match orders to reports; in some cases required by
  them. See [`ServiceRequest`](https://nw-gmsa.github.io/en/StructureDefinition-ServiceRequest.html).
- **Order Filler Number** - an identifier assigned by the Order Placer/Requesting
  Genomic Laboratory. Useful for tracking orders and reports. See
  [`ServiceRequest`](https://nw-gmsa.github.io/en/StructureDefinition-ServiceRequest.html).
- **Duo/Trio relationship details** - how a family member is related to the proband
  (mother, father, sibling...), not just that `participant_type` says "Family Member".
  `participant_type` only says Proband or Family Member. Rather than a new manifest column, the more likely
  source is NHS England: PDS's
  [`GET /Patient/{id}/RelatedPerson`](https://digital.nhs.uk/developer/api-catalogue/personal-demographics-service-fhir#get-/Patient/-id-/RelatedPerson)
  or the
  [Related Person Network FHIR API](https://digital.nhs.uk/developer/api-catalogue/related-person-network---fhir-api),
  queried by NHS number. Either could supply the specific relationship. It could then be
  coded as `RelatedPerson` resources (`MTH`, `FTH`, `SIB`... from the RelatedPerson
  Relationship Type value set).

### On making Order Number mandatory

`ServiceRequest.identifier` (Order Number) may need to become a mandatory field -
either an Order Placer Number or an Order Filler Number would be sufficient on its own.
An earlier pass used the Automation Manager's (Genomics England Limited) own
`referral_id` and `patient_ngis_id` combined as an Order Filler Number assigned by
Genomics England (ODS `8J834`), `f"{referral_id}-{patient_ngis_id}"`, which the IG
published as
the NGIS Composite Order Number (since removed from the IG).
It isn't unique per order (a cancer referral's germline and tumour rows share it), so
the worked example above now builds the Order Placer Number from `dispatched_sample_lsid`
instead, assigned by the sending lab (see the ServiceRequest section).